In [ ]:
# Cell 1 — Objective: Environment setup (requirements)
# Install ONLY the minimal Python packages used by this notebook.
# - No cloud calls, no secrets, no interactive prompts.
# - Keep it deterministic and quiet.

%pip install --quiet --upgrade pip
%pip install --quiet oci ipywidgets paramiko

# Lightweight presence check for required CLIs (no flags to avoid compatibility issues).
import shutil, sys

def _presence(name: str) -> str:
    p = shutil.which(name)
    return f"{name}: {'FOUND at ' + p if p else 'NOT FOUND'}"

print("Python:", sys.version.split()[0])
print(_presence("terraform"))
print(_presence("kubectl"))
print(_presence("oci"))  # optional; used for kubeconfig convenience if present

print("Cell 1 complete: Environment ready")
print("NEXT: Run Cell 2 — Central variables & credentials (no network calls).")


In [ ]:
# Cell 2 — Objective: Central variables & credentials (single source of truth; no network calls)
# - Centralize ALL configurable values; allow environment overrides where safe.
# - Load OCI config locally; DO NOT print secrets; NO network calls.
# - Create OUTPUT_DIR; define TS_UTC (UTC).
# - No cloud actions here.

import os
from pathlib import Path
from datetime import datetime, timezone
import oci

# --- OCI config (local read/validate; no network calls) ---
OCI_CONFIG_FILE = os.path.expanduser(os.environ.get("OCI_CONFIG_FILE", "~/.oci/config"))
OCI_PROFILE     = os.environ.get("OCI_PROFILE", "DEFAULT")

_cfg = oci.config.from_file(file_location=OCI_CONFIG_FILE, profile_name=OCI_PROFILE)
oci.config.validate_config(_cfg)  # local validation only

TENANCY_OCID = os.environ.get("TENANCY_OCID", _cfg["tenancy"])
USER_OCID    = os.environ.get("USER_OCID",    _cfg["user"])
FINGERPRINT  = os.environ.get("FINGERPRINT",  _cfg["fingerprint"])

OCI_PRIVATE_KEY_PATH   = os.path.expanduser(os.environ.get("OCI_PRIVATE_KEY_PATH", _cfg.get("key_file", "")))
PRIVATE_KEY_PASSPHRASE = os.environ.get("OCI_PASSPHRASE", os.environ.get("OCI_PRIVATE_KEY_PASSPHRASE", _cfg.get("pass_phrase", "")))
REGION                 = os.environ.get("REGION", _cfg["region"])

# Fail fast on required OCI private key file presence (do not print key content)
if not OCI_PRIVATE_KEY_PATH or not Path(OCI_PRIVATE_KEY_PATH).exists():
    raise FileNotFoundError(f"OCI private key not found. Check OCI_CONFIG_FILE/PROFILE or set OCI_PRIVATE_KEY_PATH. Current: {OCI_PRIVATE_KEY_PATH or '(unset)'}")

# --- Deployment mode & namespace ---
DEPLOY_MODE = os.environ.get("DEPLOY_MODE", "both").strip().lower()  # "generators-only" | "oke-only" | "both"
WORK_NS     = os.environ.get("WORK_NS", "lbtest").strip()

# Derived enables from DEPLOY_MODE (used later cells; no calls here)
ENABLE_OKE        = DEPLOY_MODE in ("oke-only", "both")
ENABLE_GENERATORS = DEPLOY_MODE in ("generators-only", "both")

# --- One-VCN CIDRs (UI-editable defaults) ---
CP_CIDR              = os.environ.get("CP_CIDR",             "10.0.1.0/24")
PUB_LB_CIDR          = os.environ.get("PUB_LB_CIDR",         "10.0.2.0/24")
WORKERS_PRIVATE_CIDR = os.environ.get("WORKERS_PRIVATE_CIDR","10.0.4.0/22")  # sized for nodes
PODS_CIDR            = os.environ.get("PODS_CIDR",           "10.0.64.0/18") # sized for pods
GENS_PUB_CIDR        = os.environ.get("GENS_PUB_CIDR",       "10.0.20.0/24")

# --- OKE (Backends only) ---
KUBERNETES_VERSION      = os.environ.get("KUBERNETES_VERSION", "v1.34.2")
SSH_PUBLIC_KEY_PATH     = os.path.expanduser(os.environ.get("SSH_PUBLIC_KEY_PATH", "~/.ssh/id_rsa.pub"))
BACKEND_NODE_SHAPE      = os.environ.get("BACKEND_NODE_SHAPE", "VM.Standard.E5.Flex")
BACKEND_OCPUS           = float(os.environ.get("BACKEND_OCPUS", "16"))
BACKEND_MEMORY_GB       = float(os.environ.get("BACKEND_MEMORY_GB", "64"))
MAX_PODS_PER_NODE       = int(os.environ.get("MAX_PODS_PER_NODE", "31"))
BACKEND_POOL_SIZE       = int(os.environ.get("BACKEND_POOL_SIZE", "1"))
NODE_IMAGE_OCID_BACKEND = os.environ.get("NODE_IMAGE_OCID_BACKEND", "").strip() or None  # optional OKE override

# Optional UI-persistence for image filter (dynamic in Cell 3; persisted here if set)
ORACLE_LINUX_MAJOR      = os.environ.get("ORACLE_LINUX_MAJOR", "any").strip().lower()  # "any" | "7" | "8" | "9" | "10" | ...

# --- Generators (standalone compute) ---
GENERATOR_COUNT     = int(os.environ.get("GENERATOR_COUNT", "4"))
GENERATOR_SHAPE     = os.environ.get("GENERATOR_SHAPE", "VM.Standard.E5.Flex")
GENERATOR_OCPUS     = float(os.environ.get("GENERATOR_OCPUS", "8"))
GENERATOR_MEMORY_GB = float(os.environ.get("GENERATOR_MEMORY_GB", "32"))
ENABLE_IPV6_ON_GENERATORS = os.environ.get("ENABLE_IPV6_ON_GENERATORS", "false").strip().lower() in ("1","true","yes","y")

# SSH ingress CIDRs for generators
SSH_ALLOWED_CIDR    = os.environ.get("SSH_ALLOWED_CIDR", "0.0.0.0/0").strip()
SSH_ALLOWED_V6_CIDR = os.environ.get("SSH_ALLOWED_V6_CIDR", "::/0").strip()

# SSH keys for generators (central values; validated later by UI)
_default_pub  = Path(os.path.expanduser("~/.ssh/id_rsa.pub"))
_default_priv = Path(os.path.expanduser("~/.ssh/id_rsa"))
GEN_SSH_PUBLIC_KEY_PATH  = os.path.expanduser(os.environ.get("GEN_SSH_PUBLIC_KEY_PATH",  str(_default_pub  if _default_pub.exists()  else "")))
GEN_SSH_PRIVATE_KEY_PATH = os.path.expanduser(os.environ.get("GEN_SSH_PRIVATE_KEY_PATH", str(_default_priv if _default_priv.exists() else "")))

# --- Locust & Targets (used only when generators enabled) ---
TEST_MODE                 = os.environ.get("TEST_MODE", "cps").strip().lower()  # "cps" | "throughput"
LOCUST_WAIT_TIME_SEC      = float(os.environ.get("LOCUST_WAIT_TIME_SEC", "1.0"))
LOCUST_CONNECT_TIMEOUT_MS = int(os.environ.get("LOCUST_CONNECT_TIMEOUT_MS", "8000"))
LOCUST_READ_TIMEOUT_MS    = int(os.environ.get("LOCUST_READ_TIMEOUT_MS",  "15000"))
LOCUST_VERIFY_TLS         = os.environ.get("LOCUST_VERIFY_TLS", "false").strip().lower() in ("1","true","yes","y")
UI_WEB_PORT               = int(os.environ.get("UI_WEB_PORT", "8089"))
EXTERNAL_TARGETS_TEXT     = os.environ.get("EXTERNAL_TARGETS_TEXT", "").strip()  # normalized later

# Paths and endpoints used by generator runtime (kept in SSOT here)
LOCUST_WORKDIR           = os.path.abspath(os.environ.get("LOCUST_WORKDIR", "/home/opc/locustwork"))
HEALTH_ENDPOINT_PATH     = os.environ.get("HEALTH_ENDPOINT_PATH", "/healthz")
THROUGHPUT_ENDPOINT_PATH = os.environ.get("THROUGHPUT_ENDPOINT_PATH", "/payload_100k")

# --- CPU → workers policy (authoritative) ---
# Canonical vars only:
#   WORKERS_PER_HOST: "auto" or "<int-string>"
#   CPU_RESERVE: int (only for auto)
#   MIN_WORKERS_PER_HOST: int
#   MAX_WORKERS_PER_HOST: int
WORKERS_PER_HOST      = os.environ.get("WORKERS_PER_HOST", "auto").strip().lower()
CPU_RESERVE           = int(os.environ.get("CPU_RESERVE", "1"))
MIN_WORKERS_PER_HOST  = int(os.environ.get("MIN_WORKERS_PER_HOST", "1"))
MAX_WORKERS_PER_HOST  = int(os.environ.get("MAX_WORKERS_PER_HOST", "32"))

# --- TLS for Service LB (used in Cell 6 to create ssl-certificate-secret) ---
# Force RSA (matches OCI LB default cipher policy). Do not inherit previous env override.
TLS_KEY_ALGO    = "rsa"                               # "rsa" | "ecdsa"
TLS_ECDSA_CURVE = os.environ.get("TLS_ECDSA_CURVE", "prime256v1").strip().lower()  # used only if TLS_KEY_ALGO=ecdsa
# Persist to environment for downstream cells
os.environ["TLS_KEY_ALGO"] = TLS_KEY_ALGO
os.environ["TLS_ECDSA_CURVE"] = TLS_ECDSA_CURVE

# --- Execution toggles (gated steps) ---
RUN_INFRA_APPLY       = os.environ.get("RUN_INFRA_APPLY", "true").strip().lower() in ("1","true","yes","y")
RUN_K8S_DEPLOY        = os.environ.get("RUN_K8S_DEPLOY", "true").strip().lower() in ("1","true","yes","y")
RUN_GENERATORS_PREP   = os.environ.get("RUN_GENERATORS_PREP", "true").strip().lower() in ("1","true","yes","y")
RUN_LOCUST            = os.environ.get("RUN_LOCUST", "false").strip().lower() in ("1","true","yes","y")

DESTROY_GENS_ON_APPLY    = os.environ.get("DESTROY_GENS_ON_APPLY", "false").strip().lower() in ("1","true","yes","y")
DESTROY_OKE_ON_APPLY     = os.environ.get("DESTROY_OKE_ON_APPLY",  "false").strip().lower() in ("1","true","yes","y")
DESTROY_NETWORK_ON_APPLY = os.environ.get("DESTROY_NETWORK_ON_APPLY", "false").strip().lower() in ("1","true","yes","y")

# --- Placeholders for selections made in Cell 3 (kept here for clarity; set by UI) ---
SELECTED_REGION = None
COMPARTMENT_ID  = ""
AD_A            = ""

# --- Output directory & UTC timestamp ---
OUTPUT_DIR = os.path.abspath(os.environ.get("OUTPUT_DIR", "./results"))
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
TS_UTC = os.environ.get("TS_UTC", "") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
os.environ["TS_UTC"] = TS_UTC  # persist for later cells

# Minimal, non-sensitive confirmation
print(f"Cell 2 ready: {OCI_CONFIG_FILE} [{OCI_PROFILE}] — region={REGION}, mode={DEPLOY_MODE}, ns={WORK_NS}")
print(f"TLS: key_algo={TLS_KEY_ALGO}" + (f", ecdsa_curve={TLS_ECDSA_CURVE}" if TLS_KEY_ALGO == 'ecdsa' else ""))
print("NEXT: Run Cell 3 — Configuration UI (no estimator UI; background validation on Apply).")


In [ ]:
# Cell 3 — Objective: Configuration UI (left‑aligned) + Background validation + Mode gating
# - Location (Region, Compartment, AD)
# - OKE (Backends only):
#   * OKE K8s version from CE cluster options
#   * Backend shape + Flex OCPUs/Mem (only for *.Flex)
#   * Oracle Linux major filter (Any/OL10/OL9/OL8/OL7, discovered dynamically)
#   * OKE Worker image override (CE sources → strict OKE-only → version/arch/GPU/OL-major → shape-compat)
#   * SSH public key (for OKE)
#   * Max pods/node + Backend nodes
# - Generators (Compute):
#   * Shape filter + shape dropdown (+ Flex OCPUs/Mem for *.Flex)
#   * Generator image (Oracle Linux; shape-compat validated)
#   * SSH pub (GEN) + SSH priv (GEN)
#   * Workers/host CPU policy: mode=auto/fixed, fixed count, CPU reserve, min/max
#   * Count + IPv6 toggle
# - One‑VCN CIDRs + Locust & Targets + Execution toggles
# - Capacity Estimator UI is removed; capacity is validated in the background on Apply and shown only on error.

import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML
import oci
import re
import ipaddress

# ----- CSS (left-aligned, clean layout) -----
display(HTML("""
<style>
  .nb3-container { max-width: 1100px; margin: 0 auto; }
  .nb3-section { border: 1px solid #e0e0e0; background: #f8f9fb; padding: 12px; margin: 8px 0; border-left: 4px solid #d2d7e5; }
  .nb3-title { font-weight: 600; margin: 0 0 6px 0; text-align: left; }
  .nb3-lbl { white-space: nowrap; font-weight: 500; text-align: left; }
  .issues-strip { background: #fdecea; color: #ba1a1a; padding: 6px 10px; border: 1px solid #f5c2c7; border-radius: 6px; font-size: 13px; margin: 0 0 6px 0; }
  .nb3-small { color:#555; font-size:12px; }
  .nb3-pre { font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, "Courier New"; white-space: pre; margin: 0; }
</style>
"""))

# ----- Small helpers -----
def _usable_ips(cidr: str) -> int:
    try:
        net = ipaddress.ip_network(cidr, strict=True)
        return max(0, net.num_addresses - 3)  # OCI reserves 3
    except Exception:
        return -1

def _set_dd_options_preserve(dd: widgets.Dropdown, pairs: list, prefer: str = None):
    old_val = dd.value
    dd.options = pairs if pairs else [("None", "")]
    new_vals = [v for _, v in dd.options]
    if old_val in new_vals:
        dd.value = old_val
    elif prefer in new_vals:
        dd.value = prefer
    else:
        dd.value = new_vals[0] if new_vals else ""

def row(label_text: str, control: widgets.Widget, label_width="220px") -> widgets.HBox:
    lbl = widgets.HTML(f"<span class='nb3-lbl'>{label_text}</span>",
                       layout=widgets.Layout(width=label_width, min_width=label_width))
    if hasattr(control, "layout"):
        control.layout.flex = "1 1 auto"
        control.layout.width = "auto"
    return widgets.HBox([lbl, control],
                        layout=widgets.Layout(width="100%", justify_content="flex-start", align_items="center", gap="12px"))

def section(title: str, *children: widgets.Widget) -> widgets.VBox:
    return widgets.VBox([widgets.HTML(f"<div class='nb3-title'>{title}</div>"), *children],
                        layout=widgets.Layout(width="100%", align_items="stretch"),
                        _dom_classes=['nb3-section'])

# Image name/parser helpers
_re_ol  = re.compile(r'(?:Oracle[- ]Linux|Oracle[-]?Linux|OL)[-_ ](?P<major>\d+)(?:\.(?P<minor>\d+))?', re.IGNORECASE)
_re_oke = re.compile(r'-OKE-(?:v)?(?P<oke>\d+\.\d+(?:\.\d+)?)\b', re.IGNORECASE)
_re_oke_any = re.compile(r'-OKE-', re.IGNORECASE)
_re_date = re.compile(r'-(?P<date>\d{4}\.\d{2}\.\d{2})-', re.IGNORECASE)

def _parse_image_traits(name: str):
    ol_major = None; ol_minor = None; oke = None; date = None
    m = _re_ol.search(name or "")
    if m:
        try:
            ol_major = int(m.group('major')) if m.group('major') else None
            ol_minor = int(m.group('minor')) if m.group('minor') else None
        except Exception:
            pass
    k = _re_oke.search(name or "")
    if k:
        oke = k.group('oke')
    d = _re_date.search(name or "")
    if d:
        date = d.group('date')
    lower = (name or "").lower()
    is_arm = ('-aarch64-' in lower)
    is_gpu = ('-gpu-' in lower)
    has_oke = bool(_re_oke_any.search(name or ""))
    return {"ol_major": ol_major, "ol_minor": ol_minor, "oke_version": oke, "is_arm": is_arm, "is_gpu": is_gpu, "date": date, "has_oke": has_oke}

def _shape_arch_flags(shape_name: str):
    s = (shape_name or "").lower()
    is_arm = any(tok in s for tok in ['.a1.', '.a2.', 'aarch', 'arm', 'ampere'])
    is_gpu = ('gpu' in s)
    return is_arm, is_gpu

def _norm_k8s_version(ver: str):
    v = (ver or "").strip()
    if v.startswith('v'): v = v[1:]
    parts = v.split('.')
    full = v if len(parts) >= 2 else v
    line = '.'.join(parts[:2]) if len(parts) >= 2 else v
    return full, line

# ----- OCI config + clients (region-scoped) -----
OCI_CONFIG_FILE = os.path.expanduser(os.environ.get("OCI_CONFIG_FILE", "~/.oci/config"))
OCI_PROFILE     = os.environ.get("OCI_PROFILE", "DEFAULT")
_cfg = oci.config.from_file(file_location=OCI_CONFIG_FILE, profile_name=OCI_PROFILE)
TENANCY_OCID = _cfg["tenancy"]

def cfg_for_region(region_name):
    c = dict(_cfg)
    c["region"] = region_name
    return c

def build_signer_from_cfg(cfg=_cfg):
    key_path = os.path.expanduser(cfg.get("key_file", ""))
    if not key_path or not Path(key_path).exists():
        raise FileNotFoundError(f"OCI private key not found at: {key_path}. Check OCI config.")
    return oci.signer.Signer(
        tenancy=cfg["tenancy"], user=cfg["user"], fingerprint=cfg["fingerprint"],
        private_key_file_location=key_path, pass_phrase=cfg.get("pass_phrase")
    )

def identity_client_for_current():
    cfg_r = cfg_for_region(region_dd.value)
    return oci.identity.IdentityClient(config=cfg_r, signer=build_signer_from_cfg(cfg_r))

def compute_client_for_current():
    cfg_r = cfg_for_region(region_dd.value)
    return oci.core.compute_client.ComputeClient(config=cfg_r, signer=build_signer_from_cfg(cfg_r)) if hasattr(oci.core, "compute_client") else oci.core.ComputeClient(config=cfg_r, signer=build_signer_from_cfg(cfg_r))

def container_engine_client_for_current():
    cfg_r = cfg_for_region(region_dd.value)
    return oci.container_engine.ContainerEngineClient(config=cfg_r, signer=build_signer_from_cfg(cfg_r))

# ----- Location (Region, Compartment, AD) -----
loc_error = widgets.HTML("")
try:
    idc_base = oci.identity.IdentityClient(config=_cfg, signer=build_signer_from_cfg())
    subs = sorted(idc_base.list_region_subscriptions(TENANCY_OCID).data, key=lambda r: r.region_name)
    region_options = [(r.region_name, r.region_name) for r in subs]
    default_region = os.environ.get("REGION", _cfg["region"])
    if not any(r.region_name == default_region for r in subs):
        default_region = subs[0].region_name
except Exception as e:
    region_options = [("Config error — check OCI config/key_file", "")]
    default_region = ""
    loc_error.value = f"<div class='issues-strip'>Location discovery error: {e}</div>"

region_dd = widgets.Dropdown(options=region_options, value=default_region)
reload_btn = widgets.Button(description="Reload", icon="refresh")
reset_btn  = widgets.Button(description="Reset",  icon="history")

comp_dd = widgets.Dropdown(options=[("Loading compartments…", "")], value="")
ad_dd   = widgets.Dropdown(options=[("Loading ADs…", "")], value="")

def list_compartments(idc):
    comps = oci.pagination.list_call_get_all_results(
        idc.list_compartments, TENANCY_OCID,
        compartment_id_in_subtree=True, access_level="ACCESSIBLE"
    ).data
    comps = [c for c in comps if c.lifecycle_state == "ACTIVE"]
    tenancy = idc.get_tenancy(TENANCY_OCID).data
    tenancy_name = getattr(tenancy, "name", "root-tenancy")
    return [(f"{tenancy_name} (root)", TENANCY_OCID)] + sorted([(c.name + f" ({c.description or 'no-desc'})", c.id) for c in comps], key=lambda t: t[0].lower())

def list_ads(idc):
    ads = oci.pagination.list_call_get_all_results(idc.list_availability_domains, TENANCY_OCID).data
    return sorted([ad.name for ad in ads]) or ["AD-1"]

def refresh_location(_=None):
    try:
        idc = identity_client_for_current()
        comps = list_compartments(idc)
        _set_dd_options_preserve(comp_dd, comps, prefer=TENANCY_OCID)
        ads = list_ads(idc)
        _set_dd_options_preserve(ad_dd, [(n, n) for n in ads], prefer=(ads[0] if ads else ""))
        loc_error.value = ""
    except Exception as e:
        comp_dd.options = [("Error loading compartments", "")]
        comp_dd.value = ""
        ad_dd.options = [("Error loading ADs", "")]
        ad_dd.value = ""
        loc_error.value = f"<div class='issues-strip'>Location discovery error: {e}</div>"

region_dd.observe(refresh_location, names="value")
reload_btn.on_click(refresh_location)
reset_btn.on_click(lambda _: (setattr(region_dd, "value", default_region), refresh_location()))
if default_region:
    refresh_location()

# ----- Mode & Namespace -----
DEPLOY_MODE = os.environ.get("DEPLOY_MODE", "both").strip().lower()
WORK_NS     = os.environ.get("WORK_NS", "lbtest").strip()
mode_dd = widgets.Dropdown(
    options=[("Generators only", "generators-only"), ("OKE only", "oke-only"), ("Both (OKE + Generators)", "both")],
    value=DEPLOY_MODE
)
ns_in = widgets.Text(value=WORK_NS)

# ----- One‑VCN CIDRs -----
cp_cidr_in   = widgets.Text(value=os.environ.get("CP_CIDR", "10.0.1.0/24"))
plb_cidr_in  = widgets.Text(value=os.environ.get("PUB_LB_CIDR", "10.0.2.0/24"))
wkp_cidr_in  = widgets.Text(value=os.environ.get("WORKERS_PRIVATE_CIDR","10.0.4.0/22"))
pods_cidr_in = widgets.Text(value=os.environ.get("PODS_CIDR", "10.0.64.0/18"))
gen_cidr_in  = widgets.Text(value=os.environ.get("GENS_PUB_CIDR","10.0.20.0/24"))

# ----- OKE (Backends only) -----
KUBERNETES_VERSION = os.environ.get("KUBERNETES_VERSION", "v1.34.2")
oke_k8s_dd     = widgets.Dropdown(options=[(KUBERNETES_VERSION, KUBERNETES_VERSION)], value=KUBERNETES_VERSION)
oke_k8s_reload = widgets.Button(description="Reload versions", icon="refresh")
oke_versions_info = widgets.HTML("<span class='nb3-small'></span>")

def refresh_oke_versions(_=None):
    try:
        ce = container_engine_client_for_current()
        resp = ce.get_cluster_options("all")
        data = getattr(resp, "data", resp)
        vers = getattr(data, "available_kubernetes_versions", None) or getattr(data, "kubernetes_versions", None) or []
        seen = set(); ordered = []
        for v in vers:
            if v not in seen:
                seen.add(v); ordered.append(v)
        items = [(v, v) for v in ordered] or [(KUBERNETES_VERSION, KUBERNETES_VERSION)]
        _set_dd_options_preserve(oke_k8s_dd, items, prefer=KUBERNETES_VERSION)
        oke_versions_info.value = f"<span class='nb3-small'>Loaded {len(items)} OKE versions</span>"
    except Exception:
        oke_versions_info.value = "<span class='nb3-small' style='color:#b00020'>Version load error; using current value.</span>"

oke_k8s_reload.on_click(refresh_oke_versions)
region_dd.observe(refresh_oke_versions, names="value")
comp_dd.observe(refresh_oke_versions, names="value")
refresh_oke_versions()

# Backend shape + Flex CPU/Mem
shape_filter_in   = widgets.Text(value="")
shape_loading_lbl = widgets.HTML("<span class='nb3-small'></span>")
b_shape_default   = os.environ.get("BACKEND_NODE_SHAPE","VM.Standard.E5.Flex")
b_shape_dd        = widgets.Dropdown(options=[(b_shape_default, b_shape_default)], value=b_shape_default)
b_ocpus_in        = widgets.BoundedFloatText(value=float(os.environ.get("BACKEND_OCPUS","16")), min=1, max=1024, step=1)
b_mem_in          = widgets.BoundedFloatText(value=float(os.environ.get("BACKEND_MEMORY_GB","64")), min=1, max=4096, step=1)

def _toggle_backend_flex(shape_name: str):
    is_flex = (shape_name or "").endswith(".Flex")
    b_ocpus_in.layout.display = "" if is_flex else "none"
    b_mem_in.layout.display   = "" if is_flex else "none"

# Oracle Linux major selector (dynamic)
ol_major_dd = widgets.Dropdown(options=[("Any","any")], value=os.environ.get("ORACLE_LINUX_MAJOR","any"))

# OKE Worker image override (CE-first; strict OKE-only; version/arch/GPU/OL-major; shape-compat)
b_img_dd        = widgets.Dropdown(options=[("Leave empty for default OKE image", "")], value="")
b_img_selected  = widgets.HTML("<span class='nb3-small'>Selected image: (default)</span>")
b_img_note      = widgets.HTML("<span class='nb3-small'></span>")
b_img_diag      = widgets.HTML("<span class='nb3-small'></span>")

def _update_selected_image_preview(dd: widgets.Dropdown, label_html: widgets.HTML, default_text="(default)"):
    try:
        lab = next((lab for (lab, val) in dd.options if val == dd.value), default_text)
        label_html.value = f"<span class='nb3-small'>Selected image: {lab}</span>"
    except Exception:
        label_html.value = f"<span class='nb3-small'>Selected image: {default_text}</span>"

def refresh_backend_shapes_and_images(*_):
    # Shapes
    try:
        cc = compute_client_for_current()
        shape_loading_lbl.value = "<span class='nb3-small'>loading shapes…</span>"
        shapes = oci.pagination.list_call_get_all_results(cc.list_shapes, TENANCY_OCID).data
        all_names = sorted({s.shape for s in shapes})
        filt = (shape_filter_in.value or "").strip().lower()
        names = [n for n in all_names if (filt in n.lower())] if filt else all_names
        items = [(n, n) for n in names] or [(b_shape_dd.value, b_shape_dd.value)]
        cur = b_shape_dd.value
        _set_dd_options_preserve(b_shape_dd, items, prefer=cur or b_shape_default)
        _toggle_backend_flex(b_shape_dd.value)
        shape_loading_lbl.value = "<span class='nb3-small'></span>"
        # Refresh OKE images after shape updates
        refresh_worker_images()
    except Exception:
        shape_loading_lbl.value = "<span class='nb3-small' style='color:#b00020'>shape load error</span>"

shape_filter_in.observe(refresh_backend_shapes_and_images, names="value")
b_shape_dd.observe(lambda ch: refresh_backend_shapes_and_images(), names="value")

def refresh_worker_images(*_):
    try:
        b_img_note.value = "<span class='nb3-small'></span>"
        b_img_diag.value = "<span class='nb3-small'></span>"

        ce = container_engine_client_for_current()
        cc = compute_client_for_current()

        # CE node pool options (authoritative)
        ce_sources_ok = True
        try:
            opt = ce.get_node_pool_options("all")
            data = getattr(opt, "data", opt)
            sources = getattr(data, "sources", None) or []
        except Exception:
            sources = []
            ce_sources_ok = False

        pool = []
        for src in sources:
            try:
                name = getattr(src, "source_name", None)
                img  = getattr(src, "image_id", None)
                st   = getattr(src, "source_type", None)
                if not (name and img and (st or "").upper() == "IMAGE"):
                    continue
                if "-OKE-" not in name:
                    continue
                pool.append((img, name))
            except Exception:
                continue

        # Fallback to Compute if CE returns nothing (selected comp → root) and keep only names with "-OKE-"
        comp_sel = comp_dd.value or TENANCY_OCID
        def _compute_oke_only(compartment_id) -> list:
            try:
                resp = oci.pagination.list_call_get_all_results(
                    cc.list_images, compartment_id,
                    sort_by="TIMECREATED", sort_order="DESC"
                )
                out = []
                for im in resp.data[:2000]:
                    nm = im.display_name or ""
                    if "-OKE-" in nm:
                        out.append((im.id, nm))
                return out
            except Exception:
                return []

        comp_pool = []; root_pool = []
        if not pool:
            comp_pool = _compute_oke_only(comp_sel)
            if not comp_pool:
                root_pool = _compute_oke_only(TENANCY_OCID)
        initial = pool if pool else (comp_pool if comp_pool else root_pool)

        if not initial:
            src_stat = "ok" if ce_sources_ok else "error"
            b_img_diag.value = f"<div class='issues-strip'>No OKE images found. Region={region_dd.value}. CE sources={src_stat}. Compute(selected)={len(comp_pool)} Compute(root)={len(root_pool)}. If both are 0, verify IAM policy for images.</div>"
            _set_dd_options_preserve(b_img_dd, [("Leave empty for default OKE image", "")], prefer="")
            _update_selected_image_preview(b_img_dd, b_img_selected)
            return

        # Filters
        ver_full, ver_line = _norm_k8s_version(oke_k8s_dd.value or "")
        is_arm_shape, is_gpu_shape = _shape_arch_flags(b_shape_dd.value or "")

        # Dynamic OL majors from pool
        majors = set()
        for (_img, nm) in initial:
            tr = _parse_image_traits(nm)
            if tr["ol_major"]:
                majors.add(tr["ol_major"])
        major_opts = [("Any","any")] + [(f"OL{m}", str(m)) for m in sorted(majors, reverse=True)]
        _set_dd_options_preserve(ol_major_dd, major_opts, prefer=ol_major_dd.value if ol_major_dd.value in [v for _,v in major_opts] else "any")
        if ol_major_dd.value != "any" and (not majors or int(ol_major_dd.value) not in majors):
            ol_major_dd.value = "any"
            b_img_note.value = "<span class='nb3-small'>No OKE images for selected OL major; reset to Any.</span>"

        def _accept_by_traits(nm: str) -> bool:
            tr = _parse_image_traits(nm)
            if ol_major_dd.value != "any":
                try:
                    want = int(ol_major_dd.value)
                    if tr["ol_major"] != want:
                        return False
                except Exception:
                    return False
            if is_arm_shape and not tr["is_arm"]:
                return False
            if (not is_arm_shape) and tr["is_arm"]:
                return False
            if is_gpu_shape and not tr["is_gpu"]:
                return False
            if (not is_gpu_shape) and tr["is_gpu"]:
                return False
            return True

        trait_filtered = [(img, nm) for (img, nm) in initial if _accept_by_traits(nm)]

        def _ver_full(nm: str) -> bool:
            tr = _parse_image_traits(nm); return (tr["oke_version"] == ver_full) if ver_full else False
        def _ver_line(nm: str) -> bool:
            tr = _parse_image_traits(nm)
            if not tr["oke_version"] or not ver_line:
                return False
            return tr["oke_version"].split('.')[:2] == ver_line.split('.')[:2]
        def _ver_any(nm: str) -> bool:
            return _parse_image_traits(nm)["oke_version"] is not None

        strict_full = [(img, nm) for (img, nm) in trait_filtered if _ver_full(nm)]
        strict_line = [(img, nm) for (img, nm) in trait_filtered if _ver_line(nm) and (img, nm) not in strict_full]
        relaxed_any = [(img, nm) for (img, nm) in trait_filtered if _ver_any(nm) and (img, nm) not in strict_full and (img, nm) not in strict_line]

        # Shape compatibility validation (accept exact or base for Flex)
        def _is_shape_compatible(image_id: str, shape_name: str) -> bool:
            try:
                resp = oci.pagination.list_call_get_all_results(
                    cc.list_image_shape_compatibility_entries, image_id=image_id
                )
                shapes = {e.shape for e in (resp.data or [])}
                if shape_name in shapes:
                    return True
                if shape_name.endswith(".Flex"):
                    base = shape_name[:-5]
                    if base in shapes:
                        return True
                return False
            except Exception:
                return False

        def _compat_pass(items):
            out = []
            for (img, nm) in items:
                if _is_shape_compatible(img, b_shape_dd.value or ""):
                    out.append((img, nm))
                if len(out) >= 200:
                    break
            return out

        strict_full  = _compat_pass(strict_full)
        strict_line  = _compat_pass(strict_line)
        relaxed_any  = _compat_pass(relaxed_any)

        # Build labeled dropdown (priority: exact → minor → any)
        def _mk_label(nm: str, prefix=""):
            tr = _parse_image_traits(nm)
            tags = []
            if tr["is_arm"]:
                tags.append("ARM")
            if tr["is_gpu"]:
                tags.append("GPU")
            tag_txt = f" [{' '.join(tags)}]" if tags else ""
            date = tr["date"] or ""
            return f"{prefix}{tag_txt} {nm}" + (f" — {date}" if date else "")

        pairs = [("Leave empty for default OKE image", "")]
        used = set()

        if strict_full:
            for (img, nm) in strict_full:
                if img in used: continue
                pairs.append((_mk_label(nm, f"[OKE v{_norm_k8s_version(oke_k8s_dd.value)[0]}] "), img)); used.add(img)
        if strict_line:
            for (img, nm) in strict_line:
                if img in used: continue
                pairs.append((_mk_label(nm, f"[OKE v{_norm_k8s_version(oke_k8s_dd.value)[1]}] "), img)); used.add(img)
        if relaxed_any:
            for (img, nm) in relaxed_any:
                if img in used: continue
                pairs.append((_mk_label(nm, "[OKE] "), img)); used.add(img)

        if len(pairs) == 1:
            ce_stat = "ok" if ce_sources_ok else "error"
            b_img_diag.value = f"<div class='issues-strip'>No OKE images after filters. Region={region_dd.value}. CE sources={ce_stat}. Initial={len(initial)} trait={len(trait_filtered)} full={len(strict_full)} minor={len(strict_line)} any={len(relaxed_any)}. Leaving override empty is recommended.</div>"

        _set_dd_options_preserve(b_img_dd, pairs[:201], prefer=os.environ.get("NODE_IMAGE_OCID_BACKEND","") or "")
        _update_selected_image_preview(b_img_dd, b_img_selected)

    except Exception:
        b_img_diag.value = "<div class='issues-strip'>Image load error. Verify region/credentials.</div>"
        _set_dd_options_preserve(b_img_dd, [("Leave empty for default OKE image", "")], prefer="")
        _update_selected_image_preview(b_img_dd, b_img_selected)

oke_k8s_dd.observe(refresh_worker_images, names="value")
region_dd.observe(refresh_worker_images, names="value")
comp_dd.observe(refresh_worker_images, names="value")
ol_major_dd.observe(refresh_worker_images, names="value")

# Initial load of shapes+images
refresh_backend_shapes_and_images()

# OKE SSH pub key (from ~/.ssh)
ssh_dir = Path(os.path.expanduser("~/.ssh"))
ssh_pub_candidates = sorted([str(f) for f in ssh_dir.iterdir() if f.is_file() and f.name.endswith(".pub")]) if (ssh_dir.exists() and ssh_dir.is_dir()) else []
SSH_PUBLIC_KEY_PATH = os.path.expanduser(os.environ.get("SSH_PUBLIC_KEY_PATH", ssh_pub_candidates[0] if ssh_pub_candidates else ""))
ssh_pub_oke_dd = widgets.Dropdown(options=[(p, p) for p in ssh_pub_candidates] or [("No *.pub in ~/.ssh", "")],
                                  value=(SSH_PUBLIC_KEY_PATH if SSH_PUBLIC_KEY_PATH in ssh_pub_candidates else (ssh_pub_candidates[0] if ssh_pub_candidates else "")))

# Other OKE params
max_pods_in  = widgets.BoundedIntText(value=int(os.environ.get("MAX_PODS_PER_NODE", "31")), min=1, max=512, step=1)
pool_size_in = widgets.BoundedIntText(value=int(os.environ.get("BACKEND_POOL_SIZE", "1")), min=1, max=5000, step=1)

# ----- Generators (Compute) -----
GENERATOR_COUNT     = int(os.environ.get("GENERATOR_COUNT", "4"))
GENERATOR_SHAPE     = os.environ.get("GENERATOR_SHAPE", "VM.Standard.E5.Flex")
GENERATOR_OCPUS     = float(os.environ.get("GENERATOR_OCPUS", "8"))
GENERATOR_MEMORY_GB = float(os.environ.get("GENERATOR_MEMORY_GB", "32"))
ENABLE_IPV6_ON_GENERATORS = os.environ.get("ENABLE_IPV6_ON_GENERATORS", "false").strip().lower() in ("1","true","yes","y")

g_shape_filter_in = widgets.Text(value="")
g_shape_loading   = widgets.HTML("<span class='nb3-small'></span>")
g_shape_dd        = widgets.Dropdown(options=[(GENERATOR_SHAPE, GENERATOR_SHAPE)], value=GENERATOR_SHAPE)
g_ocpus_in        = widgets.BoundedFloatText(value=GENERATOR_OCPUS, min=1, max=1024, step=1)
g_mem_in          = widgets.BoundedFloatText(value=GENERATOR_MEMORY_GB, min=1, max=4096, step=1)

def _toggle_gen_flex(shape_name: str):
    is_flex = (shape_name or "").endswith(".Flex")
    g_ocpus_in.layout.display = "" if is_flex else "none"
    g_mem_in.layout.display   = "" if is_flex else "none"

g_img_dd        = widgets.Dropdown(options=[("Select a shape first", "")], value="")
g_img_loading   = widgets.HTML("<span class='nb3-small'></span>")
g_img_selected  = widgets.HTML("<span class='nb3-small'>Selected image: (none)</span>")
g_count_in      = widgets.BoundedIntText(value=GENERATOR_COUNT, min=1, max=10000, step=1)
g_v6_cb         = widgets.Checkbox(value=ENABLE_IPV6_ON_GENERATORS)

# SSH keys (GEN)
ssh_pub_candidates_gen = ssh_pub_candidates
ssh_priv_candidates_gen = sorted([str(f) for f in ssh_dir.iterdir() if f.is_file() and not f.name.endswith(".pub")]) if (ssh_dir.exists() and ssh_dir.is_dir()) else []
GEN_SSH_PUBLIC_KEY_PATH  = os.path.expanduser(os.environ.get("GEN_SSH_PUBLIC_KEY_PATH",  (ssh_pub_candidates_gen[0] if ssh_pub_candidates_gen else "")))
GEN_SSH_PRIVATE_KEY_PATH = os.path.expanduser(os.environ.get("GEN_SSH_PRIVATE_KEY_PATH", (ssh_priv_candidates_gen[0] if ssh_priv_candidates_gen else "")))
ssh_pub_gen_dd  = widgets.Dropdown(options=[(p, p) for p in ssh_pub_candidates_gen]  or [("No *.pub in ~/.ssh", "")],
                                   value=(GEN_SSH_PUBLIC_KEY_PATH if GEN_SSH_PUBLIC_KEY_PATH in ssh_pub_candidates_gen else (ssh_pub_candidates_gen[0] if ssh_pub_candidates_gen else "")))
ssh_priv_gen_dd = widgets.Dropdown(options=[(p, p) for p in ssh_priv_candidates_gen] or [("No private keys in ~/.ssh", "")],
                                   value=(GEN_SSH_PRIVATE_KEY_PATH if GEN_SSH_PRIVATE_KEY_PATH in ssh_priv_candidates_gen else (ssh_priv_candidates_gen[0] if ssh_priv_candidates_gen else "")))

# Workers/host CPU policy (auto/fixed; canonical only)
def _default_mode_from_value(v):
    if str(v).strip().lower() == "auto":
        return "auto"
    try:
        int(str(v).strip())
        return "fixed"
    except Exception:
        return "auto"

workers_per_host_env = os.environ.get("WORKERS_PER_HOST", "auto").strip().lower() or "auto"
workers_per_host_mode_in = widgets.Dropdown(
    options=[("Auto (use OCPUs)", "auto"), ("Fixed count", "fixed")],
    value=_default_mode_from_value(workers_per_host_env),
    description=""
)
fixed_workers_default = 1
if _default_mode_from_value(workers_per_host_env) == "fixed":
    try:
        fixed_workers_default = max(1, int(workers_per_host_env))
    except Exception:
        fixed_workers_default = 1
fixed_workers_in = widgets.BoundedIntText(value=fixed_workers_default, min=1, max=1024, step=1)
cpu_reserve_in   = widgets.BoundedIntText(value=int(os.environ.get("CPU_RESERVE","1")), min=0, max=16, step=1)
min_workers_in   = widgets.BoundedIntText(value=int(os.environ.get("MIN_WORKERS_PER_HOST","1")), min=1, max=1024, step=1)
max_workers_in   = widgets.BoundedIntText(value=int(os.environ.get("MAX_WORKERS_PER_HOST","32")), min=1, max=2048, step=1)

lbl_err_workers = widgets.HTML("")

def _toggle_fixed_inputs(_=None):
    fixed_workers_in.disabled = (workers_per_host_mode_in.value != "fixed")
_toggle_fixed_inputs()
workers_per_host_mode_in.observe(_toggle_fixed_inputs, names="value")

def refresh_gen_shapes_and_images(*_):
    try:
        cc = compute_client_for_current()
        # Shapes
        g_shape_loading.value = "<span class='nb3-small'>loading shapes…</span>"
        shapes = oci.pagination.list_call_get_all_results(cc.list_shapes, TENANCY_OCID).data
        all_names = sorted({s.shape for s in shapes})
        filt = (g_shape_filter_in.value or "").strip().lower()
        names = [n for n in all_names if (filt in n.lower())] if filt else all_names
        items = [(n, n) for n in names] or [(g_shape_dd.value, g_shape_dd.value)]
        cur = g_shape_dd.value
        _set_dd_options_preserve(g_shape_dd, items, prefer=cur or GENERATOR_SHAPE)
        _toggle_gen_flex(g_shape_dd.value)
        g_shape_loading.value = "<span class='nb3-small'></span>"

        # Generator images: list in root, then validate shape compatibility
        g_img_loading.value = "<span class='nb3-small'>loading images…</span>"
        imgs = []
        try:
            resp = oci.pagination.list_call_get_all_results(
                cc.list_images, TENANCY_OCID,
                operating_system="Oracle Linux", sort_by="TIMECREATED", sort_order="DESC"
            )
            imgs = list(resp.data)[:2000]
        except Exception:
            imgs = []
        pairs = []
        used = 0
        for im in imgs:
            if used >= 200:
                break
            try:
                resp = oci.pagination.list_call_get_all_results(
                    cc.list_image_shape_compatibility_entries, image_id=im.id
                )
                shapes = {e.shape for e in (resp.data or [])}
                ok = False
                if g_shape_dd.value in shapes:
                    ok = True
                elif g_shape_dd.value.endswith(".Flex") and g_shape_dd.value[:-5] in shapes:
                    ok = True
                if not ok:
                    continue
                try:
                    created = im.time_created.strftime("%Y-%m-%d")
                except Exception:
                    created = ""
                pairs.append((f"{im.display_name} — {im.operating_system} {im.operating_system_version}" + (f" — {created}" if created else ""), im.id))
                used += 1
            except Exception:
                continue

        _set_dd_options_preserve(g_img_dd, pairs or [("No compatible Oracle Linux images for selected shape", "")], prefer=os.environ.get("GEN_IMAGE_ID","") or (pairs[0][1] if pairs else ""))
        # Preview
        try:
            lab = next((lab for (lab, val) in g_img_dd.options if val == g_img_dd.value), "(none)")
        except Exception:
            lab = "(none)"
        g_img_selected.value = f"<span class='nb3-small'>Selected image: {lab}</span>"
        g_img_loading.value = "<span class='nb3-small'></span>"
    except Exception:
        g_shape_loading.value = "<span class='nb3-small' style='color:#b00020'>shape load error</span>"
        g_img_loading.value   = "<span class='nb3-small' style='color:#b00020'>image load error</span>"

g_shape_filter_in.observe(refresh_gen_shapes_and_images, names="value")
g_shape_dd.observe(lambda ch: (_toggle_gen_flex(g_shape_dd.value), refresh_gen_shapes_and_images()), names="value")
region_dd.observe(refresh_gen_shapes_and_images, names="value")
comp_dd.observe(refresh_gen_shapes_and_images, names="value")
refresh_gen_shapes_and_images()

# ----- Locust & Targets -----
mode_locust_dd  = widgets.Dropdown(options=[("CPS (connections/sec)", "cps"), ("Throughput (GET payload)", "throughput")],
                                   value=os.environ.get("TEST_MODE", "cps"))
wait_in         = widgets.BoundedFloatText(value=float(os.environ.get("LOCUST_WAIT_TIME_SEC","1.0")), min=0.0, max=60.0, step=0.1)
conn_ms_in      = widgets.BoundedIntText(value=int(os.environ.get("LOCUST_CONNECT_TIMEOUT_MS","8000")), min=100, max=60000, step=100)
read_ms_in      = widgets.BoundedIntText(value=int(os.environ.get("LOCUST_READ_TIMEOUT_MS","15000")), min=100, max=120000, step=100)
verify_tls_cb   = widgets.Checkbox(value=os.environ.get("LOCUST_VERIFY_TLS","false").strip().lower() in ("1","true","yes","y"))
ui_port_in      = widgets.BoundedIntText(value=int(os.environ.get("UI_WEB_PORT","8089")), min=1024, max=65535, step=1)
targets_in      = widgets.Textarea(value=os.environ.get("EXTERNAL_TARGETS_TEXT",""), layout=widgets.Layout(height="80px"))
targets_prev    = widgets.HTML("<span class='nb3-small'>Normalized: (none)</span>")
targets_err     = widgets.HTML("")

def _normalize_targets(txt: str) -> list[str]:
    out = []
    for tok in (txt or "").split(","):
        t = (tok or "").strip()
        if not t:
            continue
        if not (t.startswith("http://") or t.startswith("https://")):
            t = "https://" + t
        try:
            scheme, rest = t.split("://", 1)
            host = rest.split("/")[0]
            if ":" in host and not (host.startswith("[") and host.endswith("]")):
                t = f"{scheme}://[{host}]" + rest[len(host):]
        except Exception:
            pass
        if t not in out:
            out.append(t)
    return out

def _update_targets_preview(*_):
    norm = _normalize_targets(targets_in.value or "")
    targets_prev.value = "<span class='nb3-small'>Normalized: " + (", ".join(norm) if norm else "(none)") + "</span>"

targets_in.observe(_update_targets_preview, names="value")
_update_targets_preview()

# ----- Execution & Destroy -----
RUN_INFRA_APPLY       = os.environ.get("RUN_INFRA_APPLY", "true").strip().lower() in ("1","true","yes","y")
RUN_K8S_DEPLOY        = os.environ.get("RUN_K8S_DEPLOY", "true").strip().lower() in ("1","true","yes","y")
RUN_GENERATORS_PREP   = os.environ.get("RUN_GENERATORS_PREP", "true").strip().lower() in ("1","true","yes","y")
RUN_LOCUST            = os.environ.get("RUN_LOCUST", "false").strip().lower() in ("1","true","yes","y")
DESTROY_GENS_ON_APPLY    = os.environ.get("DESTROY_GENS_ON_APPLY", "false").strip().lower() in ("1","true","yes","y")
DESTROY_OKE_ON_APPLY     = os.environ.get("DESTROY_OKE_ON_APPLY",  "false").strip().lower() in ("1","true","yes","y")
DESTROY_NETWORK_ON_APPLY = os.environ.get("DESTROY_NETWORK_ON_APPLY", "false").strip().lower() in ("1","true","yes","y")

run_infra_cb   = widgets.Checkbox(value=RUN_INFRA_APPLY, description="Run: Infra apply")
run_k8s_cb     = widgets.Checkbox(value=RUN_K8S_DEPLOY, description="Run: K8s deploy (TLS+NGINX+LB)")
run_prep_cb    = widgets.Checkbox(value=RUN_GENERATORS_PREP, description="Run: Generators prepare (pip+tmux)")
run_locust_cb  = widgets.Checkbox(value=RUN_LOCUST, description="Run: Locust (master+workers)")
destroy_gens_cb= widgets.Checkbox(value=DESTROY_GENS_ON_APPLY, description="Destroy Generators on apply")
destroy_oke_cb = widgets.Checkbox(value=DESTROY_OKE_ON_APPLY, description="Destroy OKE on apply")
destroy_net_cb = widgets.Checkbox(value=DESTROY_NETWORK_ON_APPLY, description="Destroy Network on apply (only when both disabled)")

# ----- Issues strip & Apply -----
issues_strip = widgets.HTML("")
apply_btn = widgets.Button(description="Apply", button_style="primary", icon="check",
                           layout=widgets.Layout(width="240px", height="36px"))
summary_out = widgets.Output()

def _capacity_validate():
    # background capacity validation; only show on error
    w_cidr = (wkp_cidr_in.value or "").strip()
    p_cidr = (pods_cidr_in.value or "").strip()
    try:
        mpp   = max(1, int(max_pods_in.value))
        nodes = max(1, int(pool_size_in.value))
    except Exception:
        return "Invalid max pods/node or node count."
    usable_workers = _usable_ips(w_cidr)
    usable_pods    = _usable_ips(p_cidr)
    if usable_workers < 0 or usable_pods < 0:
        return "Invalid workers_private or pods CIDR."
    sys_p = 64  # hidden system pod budget
    node_cap_by_pods = (usable_pods - sys_p) // mpp if usable_pods > sys_p else 0
    effective_nodes  = min(usable_workers, node_cap_by_pods)
    total_pods_needed = nodes * mpp + sys_p
    if nodes > effective_nodes:
        return "Insufficient node capacity: increase workers_private and/or pods CIDR or reduce Backend nodes."
    if total_pods_needed > usable_pods:
        return "Pods subnet too small: increase pods CIDR or reduce max pods/node or node count."
    return ""

def on_apply_clicked(_b):
    summary_out.clear_output()
    issues_strip.value = ""
    targets_err.value = ""

    errs = []

    # Generators validation (when enabled)
    gens_enabled = mode_dd.value in ("generators-only", "both")
    if gens_enabled:
        # Require generator image
        if not (g_img_dd.value or "").strip():
            errs.append("Generators image not selected")
        # SSH keys must be present and exist
        if not (ssh_pub_gen_dd.value or "").strip() or not Path(os.path.expanduser(ssh_pub_gen_dd.value)).exists():
            errs.append("GEN SSH public key not found")
        if not (ssh_priv_gen_dd.value or "").strip() or not Path(os.path.expanduser(ssh_priv_gen_dd.value)).exists():
            errs.append("GEN SSH private key not found")
        # Workers policy min/max
        try:
            mn = int(min_workers_in.value)
            mx = int(max_workers_in.value)
            if mn > mx:
                errs.append("Min workers/host cannot exceed Max workers/host")
        except Exception:
            errs.append("Workers/host values must be integers")
        # Targets required in generators-only mode
        if mode_dd.value == "generators-only":
            norm = _normalize_targets(targets_in.value or "")
            if not norm:
                targets_err.value = "<div class='issues-strip'>Targets required for generators-only mode.</div>"
                errs.append("Missing targets")

    # OKE capacity gate (when enabled) — background only
    if mode_dd.value in ("oke-only", "both"):
        cap_msg = _capacity_validate()
        if cap_msg:
            errs.append(cap_msg)

    if errs:
        issues_strip.value = "<div class='issues-strip'>Apply aborted: " + "; ".join(errs) + "</div>"
        return

    # Persist to env — Location
    os.environ["REGION"]         = region_dd.value
    os.environ["COMPARTMENT_ID"] = comp_dd.value or ""
    os.environ["AD_A"]           = ad_dd.value or ""

    # Mode & Namespace
    os.environ["DEPLOY_MODE"] = mode_dd.value
    os.environ["WORK_NS"]     = ns_in.value.strip() or "lbtest"

    # VCN
    os.environ["CP_CIDR"]              = (cp_cidr_in.value or "").strip()
    os.environ["PUB_LB_CIDR"]          = (plb_cidr_in.value or "").strip()
    os.environ["WORKERS_PRIVATE_CIDR"] = (wkp_cidr_in.value or "").strip()
    os.environ["PODS_CIDR"]            = (pods_cidr_in.value or "").strip()
    os.environ["GENS_PUB_CIDR"]        = (gen_cidr_in.value or "").strip()

    # OKE
    os.environ["KUBERNETES_VERSION"]  = (oke_k8s_dd.value or "").strip()
    os.environ["SSH_PUBLIC_KEY_PATH"] = os.path.expanduser(ssh_pub_oke_dd.value or "")
    os.environ["BACKEND_NODE_SHAPE"]  = (b_shape_dd.value or "").strip()
    os.environ["BACKEND_OCPUS"]       = str(float(b_ocpus_in.value))
    os.environ["BACKEND_MEMORY_GB"]   = str(float(b_mem_in.value))
    os.environ["MAX_PODS_PER_NODE"]   = str(int(max_pods_in.value))
    os.environ["BACKEND_POOL_SIZE"]   = str(int(pool_size_in.value))
    os.environ["NODE_IMAGE_OCID_BACKEND"] = (b_img_dd.value or "").strip()
    os.environ["ORACLE_LINUX_MAJOR"]      = (ol_major_dd.value or "any").replace("OL","").strip().lower()

    # Generators
    os.environ["GENERATOR_COUNT"]     = str(int(g_count_in.value))
    os.environ["GENERATOR_SHAPE"]     = (g_shape_dd.value or "").strip()
    os.environ["GENERATOR_OCPUS"]     = str(float(g_ocpus_in.value))
    os.environ["GENERATOR_MEMORY_GB"] = str(float(g_mem_in.value))
    os.environ["ENABLE_IPV6_ON_GENERATORS"] = "true" if bool(g_v6_cb.value) else "false"
    os.environ["GEN_IMAGE_ID"]        = (g_img_dd.value or "").strip()
    os.environ["GEN_SSH_PUBLIC_KEY_PATH"]  = os.path.expanduser(ssh_pub_gen_dd.value or "")
    os.environ["GEN_SSH_PRIVATE_KEY_PATH"] = os.path.expanduser(ssh_priv_gen_dd.value or "")
    # Also set generic SSH_* for downstream self-contained steps
    os.environ["SSH_PUBLIC_KEY_PATH"]  = os.environ["GEN_SSH_PUBLIC_KEY_PATH"]
    os.environ["SSH_PRIVATE_KEY_PATH"] = os.environ["GEN_SSH_PRIVATE_KEY_PATH"]

    # Workers/host CPU policy (canonical only)
    mode = (workers_per_host_mode_in.value or "auto").strip().lower()
    if mode == "auto":
        os.environ["WORKERS_PER_HOST"] = "auto"
    else:
        os.environ["WORKERS_PER_HOST"] = str(int(fixed_workers_in.value))
    os.environ["CPU_RESERVE"]          = str(int(cpu_reserve_in.value))
    os.environ["MIN_WORKERS_PER_HOST"] = str(int(min_workers_in.value))
    os.environ["MAX_WORKERS_PER_HOST"] = str(int(max_workers_in.value))

    # Locust & Targets
    os.environ["TEST_MODE"]                 = (mode_locust_dd.value or "cps").strip().lower()
    os.environ["LOCUST_WAIT_TIME_SEC"]      = str(float(wait_in.value))
    os.environ["LOCUST_CONNECT_TIMEOUT_MS"] = str(int(conn_ms_in.value))
    os.environ["LOCUST_READ_TIMEOUT_MS"]    = str(int(read_ms_in.value))
    os.environ["LOCUST_VERIFY_TLS"]         = "true" if bool(verify_tls_cb.value) else "false"
    os.environ["UI_WEB_PORT"]               = str(int(ui_port_in.value))
    os.environ["EXTERNAL_TARGETS_TEXT"]     = ",".join(_normalize_targets(targets_in.value or ""))

    # Paths/endpoints (persist from SSOT; no UI in this cell)
    os.environ["LOCUST_WORKDIR"]           = os.environ.get("LOCUST_WORKDIR", "/home/opc/locustwork")
    os.environ["HEALTH_ENDPOINT_PATH"]     = os.environ.get("HEALTH_ENDPOINT_PATH", "/healthz")
    os.environ["THROUGHPUT_ENDPOINT_PATH"] = os.environ.get("THROUGHPUT_ENDPOINT_PATH", "/payload_100k")

    # Toggles
    os.environ["RUN_INFRA_APPLY"]     = "true"  if bool(run_infra_cb.value) else "false"
    os.environ["RUN_K8S_DEPLOY"]      = "true"  if bool(run_k8s_cb.value)   else "false"
    os.environ["RUN_GENERATORS_PREP"] = "true"  if bool(run_prep_cb.value)  else "false"
    os.environ["RUN_LOCUST"]          = "true"  if bool(run_locust_cb.value)else "false"
    os.environ["DESTROY_GENS_ON_APPLY"]    = "true" if bool(destroy_gens_cb.value) else "false"
    os.environ["DESTROY_OKE_ON_APPLY"]     = "true" if bool(destroy_oke_cb.value)  else "false"
    os.environ["DESTROY_NETWORK_ON_APPLY"] = "true" if bool(destroy_net_cb.value)  else "false"

    # Summary
    lines = []
    lines.append(f"region={os.environ['REGION']}  compartment={os.environ['COMPARTMENT_ID']}  ad={os.environ['AD_A']}")
    lines.append(f"mode={os.environ['DEPLOY_MODE']}  ns={os.environ['WORK_NS']}")
    lines.append(f"VCN: cp={os.environ['CP_CIDR']} pub_lb={os.environ['PUB_LB_CIDR']} workers_private={os.environ['WORKERS_PRIVATE_CIDR']} pods={os.environ['PODS_CIDR']} gens_pub={os.environ['GENS_PUB_CIDR']}")
    lines.append(f"OKE: k8s={os.environ['KUBERNETES_VERSION']} shape={os.environ['BACKEND_NODE_SHAPE']} ocpus={os.environ['BACKEND_OCPUS']} memGB={os.environ['BACKEND_MEMORY_GB']} max_pods/node={os.environ['MAX_PODS_PER_NODE']} backend_nodes={os.environ['BACKEND_POOL_SIZE']}")
    lines.append(f"Worker image override={(os.environ['NODE_IMAGE_OCID_BACKEND'] or '(default)')}, OL_major={os.environ['ORACLE_LINUX_MAJOR']}")
    lines.append(f"GEN: count={os.environ['GENERATOR_COUNT']} shape={os.environ['GENERATOR_SHAPE']} ocpus={os.environ['GENERATOR_OCPUS']} memGB={os.environ['GENERATOR_MEMORY_GB']} v6={os.environ['ENABLE_IPV6_ON_GENERATORS']}")
    # Workers policy summary
    wph = os.environ.get("WORKERS_PER_HOST","auto")
    if wph == "auto":
        lines.append(f"Workers policy: mode=auto | reserve={os.environ.get('CPU_RESERVE')} | min={os.environ.get('MIN_WORKERS_PER_HOST')} | max={os.environ.get('MAX_WORKERS_PER_HOST')}")
    else:
        lines.append(f"Workers policy: mode=fixed | fixed={wph}")
    lines.append(f"Locust: mode={os.environ['TEST_MODE']} wait(s)={os.environ['LOCUST_WAIT_TIME_SEC']} connect(ms)={os.environ['LOCUST_CONNECT_TIMEOUT_MS']} read(ms)={os.environ['LOCUST_READ_TIMEOUT_MS']} verify_tls={os.environ['LOCUST_VERIFY_TLS']} ui_port={os.environ['UI_WEB_PORT']}")
    lines.append(f"targets={(os.environ['EXTERNAL_TARGETS_TEXT'] or '(none)')}")
    lines.append(f"toggles: run_infra={os.environ['RUN_INFRA_APPLY']} run_k8s={os.environ['RUN_K8S_DEPLOY']} run_prep={os.environ['RUN_GENERATORS_PREP']} run_locust={os.environ['RUN_LOCUST']} destroy_gens={os.environ['DESTROY_GENS_ON_APPLY']} destroy_oke={os.environ['DESTROY_OKE_ON_APPLY']} destroy_net={os.environ['DESTROY_NETWORK_ON_APPLY']}")

    summary_out.clear_output()
    with summary_out:
        display(widgets.HTML(value="<b>Selections applied</b>"))
        display(widgets.HTML(value=f"<div class='nb3-pre'>{'<br/>'.join(lines)}</div>"))
    print("Cell 3 complete: Selections applied.")

apply_btn.on_click(on_apply_clicked)

# ----- Compose sections (no Capacity Estimator UI; validation runs on Apply only) -----
sec_location = section("Location",
    loc_error,
    row("Region:", region_dd), widgets.HBox([reload_btn, reset_btn], layout=widgets.Layout(gap="8px")),
    row("Compartment:", comp_dd),
    row("Availability Domain:", ad_dd)
)

sec_mode = section("Mode & Namespace",
    row("Deploy mode:", mode_dd),
    row("K8s namespace:", ns_in)
)

sec_vcn  = section("One‑VCN Networking",
    row("cp CIDR:",   cp_cidr_in),
    row("pub_lb CIDR:", plb_cidr_in),
    row("workers_private CIDR:", wkp_cidr_in),
    row("pods CIDR:", pods_cidr_in),
    row("gens_pub CIDR:", gen_cidr_in)
)

sec_oke  = section("OKE (Backends only)",
    row("OKE K8s version:", oke_k8s_dd), widgets.HBox([oke_k8s_reload, oke_versions_info], layout=widgets.Layout(gap="8px")),
    row("Shape filter:", shape_filter_in), shape_loading_lbl,
    row("Backend shape:", b_shape_dd),
    row("Backend OCPUs:", b_ocpus_in),
    row("Backend Memory (GB):", b_mem_in),
    row("Oracle Linux major:", ol_major_dd),
    row("Worker image (override):", b_img_dd), b_img_selected, b_img_note, b_img_diag,
    row("SSH pub (OKE):", ssh_pub_oke_dd),
    row("Max pods/node:", max_pods_in),
    row("Backend nodes:", pool_size_in)
)

sec_gen  = section("Generators (Compute)",
    row("Shape filter:", g_shape_filter_in), g_shape_loading,
    row("Generator shape:", g_shape_dd),
    row("Generator OCPUs:", g_ocpus_in),
    row("Generator Memory (GB):", g_mem_in),
    row("Generator image:", g_img_dd), g_img_selected, g_img_loading,
    row("SSH pub (GEN):", ssh_pub_gen_dd),
    row("SSH priv (GEN):", ssh_priv_gen_dd),
    row("Workers/host mode:", workers_per_host_mode_in),
    row("Fixed workers/host:", fixed_workers_in),
    row("CPU reserve (auto):", cpu_reserve_in),
    row("Min workers/host:", min_workers_in),
    row("Max workers/host:", max_workers_in),
    lbl_err_workers,
    row("Generators count:", g_count_in),
    row("Assign IPv6 to generators:", g_v6_cb)
)

sec_loc  = section("Locust & Targets",
    row("Locust mode:", mode_locust_dd),
    row("Wait(s)/user:", wait_in),
    row("Connect ms:",   conn_ms_in),
    row("Read ms:",      read_ms_in),
    row("Verify TLS:",   verify_tls_cb),
    row("UI port:",      ui_port_in),
    row("External targets:", targets_in), targets_prev, targets_err
)

# Mode-based visibility
def _toggle_by_mode(*_):
    is_oke = (mode_dd.value in ("oke-only", "both"))
    sec_oke.layout.display = "" if is_oke else "none"
    is_gen = (mode_dd.value in ("generators-only", "both"))
    sec_gen.layout.display = "" if is_gen else "none"

_toggle_by_mode()
mode_dd.observe(_toggle_by_mode, names="value")

container = widgets.VBox(
    [issues_strip,
     sec_location, sec_mode, sec_vcn, sec_oke, sec_gen, sec_loc,
     section("Execution & Destroy",
         run_infra_cb, run_k8s_cb, run_prep_cb, run_locust_cb,
         destroy_gens_cb, destroy_oke_cb, destroy_net_cb
     ),
     widgets.HBox([apply_btn], layout=widgets.Layout(justify_content="center")),
     summary_out],
    layout=widgets.Layout(width="100%", align_items="stretch", justify_content="flex-start"),
    _dom_classes=['nb3-container']
)

display(container)
print("Cell 3 loaded: Region-scoped OKE images (CE sources), version/shape/arch/OL-major filtered. Configure, then click Apply.")


In [ ]:
# Cell 4 — Objective: Generate combined Terraform stack (One‑VCN + conditional OKE backend + conditional Generators) and cloud‑init
# - Writes ./stack with: provider.tf, variables.tf, locals.tf, network.tf, oke.tf, generators.tf, outputs.tf, terraform.tfvars
# - Writes ./stack/cloud-init/generator.sh (tuning-only; no package installs) including a multi-capable locustfile.py
# - Honors DEPLOY_MODE from Cell 3 → enable_oke/enable_generators
# - Uses ONLY centralized values from Cell 2/3 (read from os.environ). No hardcoded secrets or paths here.
# - Security model: FULL STATELESS posture — Default Security List and all NSGs have symmetric stateless rules for IPv4 and IPv6
#   (protocol="all", stateless=true) in both directions, ensuring explicit return paths for stateless flows.

import os
from pathlib import Path
import textwrap

def _fail(msg: str):
    raise ValueError(msg)

# Read authoritative config from environment (set by Cell 2 and persisted by Cell 3 Apply)
OCI_PROFILE          = os.environ.get("OCI_PROFILE", "DEFAULT").strip()
REGION               = os.environ.get("REGION", "").strip()
COMPARTMENT_ID       = os.environ.get("COMPARTMENT_ID", "").strip()
AD_A                 = os.environ.get("AD_A", "").strip()

DEPLOY_MODE          = os.environ.get("DEPLOY_MODE", "both").strip().lower()
WORK_NS              = os.environ.get("WORK_NS", "lbtest").strip()

# One‑VCN CIDRs
CP_CIDR              = os.environ.get("CP_CIDR", "10.0.1.0/24").strip()
PUB_LB_CIDR          = os.environ.get("PUB_LB_CIDR", "10.0.2.0/24").strip()
WORKERS_PRIVATE_CIDR = os.environ.get("WORKERS_PRIVATE_CIDR", "10.0.4.0/22").strip()
PODS_CIDR            = os.environ.get("PODS_CIDR", "10.0.64.0/18").strip()
GENS_PUB_CIDR        = os.environ.get("GENS_PUB_CIDR", "10.0.20.0/24").strip()

# OKE (Backends only)
KUBERNETES_VERSION   = os.environ.get("KUBERNETES_VERSION", "v1.34.2").strip()
SSH_PUBLIC_KEY_PATH_OKE = os.path.expanduser(os.environ.get("SSH_PUBLIC_KEY_PATH", ""))  # used by node pool
BACKEND_NODE_SHAPE   = os.environ.get("BACKEND_NODE_SHAPE", "VM.Standard.E5.Flex").strip()
BACKEND_OCPUS        = os.environ.get("BACKEND_OCPUS", "16").strip()
BACKEND_MEMORY_GB    = os.environ.get("BACKEND_MEMORY_GB", "64").strip()
MAX_PODS_PER_NODE    = os.environ.get("MAX_PODS_PER_NODE", "31").strip()
BACKEND_POOL_SIZE    = os.environ.get("BACKEND_POOL_SIZE", "1").strip()
NODE_IMAGE_OCID_BACKEND = os.environ.get("NODE_IMAGE_OCID_BACKEND", "").strip()  # required for OKE in this stack

# Generators (Compute)
GENERATOR_COUNT      = os.environ.get("GENERATOR_COUNT", "1").strip()
GENERATOR_SHAPE      = os.environ.get("GENERATOR_SHAPE", "VM.Standard.E5.Flex").strip()
GENERATOR_OCPUS      = os.environ.get("GENERATOR_OCPUS", "8").strip()
GENERATOR_MEMORY_GB  = os.environ.get("GENERATOR_MEMORY_GB", "32").strip()
ENABLE_IPV6_ON_GENERATORS = os.environ.get("ENABLE_IPV6_ON_GENERATORS", "false").strip().lower() in ("1","true","yes","y")
GEN_IMAGE_ID         = os.path.expanduser(os.environ.get("GEN_IMAGE_ID", "")).strip()  # generator image (Oracle Linux) chosen in Cell 3

# SSH keys for Generators (authorized_keys content + path to private key for later SSH-based steps)
GEN_SSH_PUBLIC_KEY_PATH  = os.path.expanduser(os.environ.get("GEN_SSH_PUBLIC_KEY_PATH", "")).strip()
GEN_SSH_PRIVATE_KEY_PATH = os.path.expanduser(os.environ.get("GEN_SSH_PRIVATE_KEY_PATH", "")).strip()

# Compute enable flags from DEPLOY_MODE (authoritative)
ENABLE_OKE        = DEPLOY_MODE in ("oke-only", "both")
ENABLE_GENERATORS = DEPLOY_MODE in ("generators-only", "both")

# Basic validations (fail-fast; NO secrets printed)
if not REGION:              _fail("REGION is empty. Ensure Cell 3 Apply persisted it.")
if not COMPARTMENT_ID:      _fail("COMPARTMENT_ID is empty. Ensure Cell 3 Apply persisted it.")
if not AD_A:                _fail("AD_A is empty. Ensure Cell 3 Apply persisted it.")
if ENABLE_OKE and (not SSH_PUBLIC_KEY_PATH_OKE or not Path(SSH_PUBLIC_KEY_PATH_OKE).exists()):
    _fail(f"OKE SSH public key not found. Current: {SSH_PUBLIC_KEY_PATH_OKE or '(unset)'}")
if ENABLE_OKE and not NODE_IMAGE_OCID_BACKEND:
    _fail("NODE_IMAGE_OCID_BACKEND is empty. Provide an OKE worker image override (Cell 3).")
if ENABLE_GENERATORS:
    if not GEN_SSH_PUBLIC_KEY_PATH or not Path(GEN_SSH_PUBLIC_KEY_PATH).exists():
        _fail(f"GEN SSH public key not found. Current: {GEN_SSH_PUBLIC_KEY_PATH or '(unset)'}")
    if not GEN_SSH_PRIVATE_KEY_PATH or not Path(GEN_SSH_PRIVATE_KEY_PATH).exists():
        _fail(f"GEN SSH private key not found. Current: {GEN_SSH_PRIVATE_KEY_PATH or '(unset)'}")
    with open(GEN_SSH_PUBLIC_KEY_PATH, "r") as f:
        GEN_SSH_PUBLIC_KEY_CONTENT = (f.read() or "").strip()
    if not GEN_SSH_PUBLIC_KEY_CONTENT:
        _fail("GEN SSH public key file is empty.")
    if not GEN_IMAGE_ID:
        _fail("GEN_IMAGE_ID (generator image OCID) is empty. Select a generator image in Cell 3.")
else:
    GEN_SSH_PUBLIC_KEY_CONTENT = ""  # unused in this mode

# Prepare directories
stack_dir = Path("stack")
cloud_init_dir = stack_dir / "cloud-init"
stack_dir.mkdir(parents=True, exist_ok=True)
cloud_init_dir.mkdir(parents=True, exist_ok=True)

# --------- Terraform files content ---------

provider_tf = textwrap.dedent(f"""
terraform {{
  required_providers {{
    oci = {{
      source  = "oracle/oci"
      version = ">= 7.30.0"
    }}
    random = {{
      source  = "hashicorp/random"
      version = ">= 3.4.3"
    }}
  }}
}}

provider "oci" {{
  config_file_profile = var.oci_profile
  region              = var.region
}}
""").strip("\n")

variables_tf = textwrap.dedent("""
# Global/auth
variable "oci_profile" { type = string }
variable "region"      { type = string }
variable "compartment_id" { type = string }
variable "ad_a"        { type = string }

# Feature toggles
variable "enable_oke"        { type = bool }
variable "enable_generators" { type = bool }

# One‑VCN CIDRs
variable "cp_cidr"              { type = string }
variable "pub_lb_cidr"          { type = string }
variable "workers_private_cidr" { type = string }
variable "pods_cidr"            { type = string }
variable "gens_pub_cidr"        { type = string }

# OKE (Backends only)
variable "kubernetes_version"      { type = string }
variable "ssh_public_key_path_oke" { type = string }
variable "backend_node_shape"      { type = string }
variable "backend_ocpus"           { type = number }
variable "backend_memory_gb"       { type = number }
variable "max_pods_per_node"       { type = number }
variable "backend_pool_size"       { type = number }
variable "node_image_ocid_backend" { type = string }

# Generators (Compute)
variable "generator_count"        { type = number }
variable "generator_shape"        { type = string }
variable "generator_ocpus"        { type = number }
variable "generator_memory_gb"    { type = number }
variable "enable_ipv6_on_generators" { type = bool }
variable "generator_image_id"     { type = string }
variable "ssh_public_key_content_gen" { type = string }
variable "ssh_private_key_path_gen"   { type = string }

# Misc
variable "bastion_plugin_name" {
  type    = string
  default = "Bastion"
}
""").strip("\n")

locals_tf = textwrap.dedent("""
resource "random_string" "state" {
  length  = 6
  upper   = false
  numeric = false
  special = false
}

locals {
  # Name prefix for resources (short, stable)
  name          = "cps-${random_string.state.result}"
  vcn_dns_label = "cps${random_string.state.result}"
}
""").strip("\n")

# Network + NSGs (TEST MODEL: COMPLETE STATELESS posture)
network_tf = textwrap.dedent("""
# network.tf — TEST MODEL (stateless “all/all”) with explicit return path rules
# - Default Security List: protocol="all" stateless egress+ingress (v4+v6) for complete stateless posture.
# - NSGs (lb/backends/generators): symmetric stateless egress+ingress (v4+v6) to avoid constraining return paths.
# - Private RT: NAT for IPv4 only (no ::/0), public RT: v4+v6 via IGW.

# VCN dual-stack (test harness)
resource "oci_core_vcn" "vcn" {
  cidr_block     = "10.0.0.0/16"  # supernet anchor (not used for routing directly)
  compartment_id = var.compartment_id
  display_name   = local.name
  is_ipv6enabled = true
  dns_label      = local.vcn_dns_label
}

# Route tables
resource "oci_core_internet_gateway" "igw" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "${local.name}-igw"
}

resource "oci_core_nat_gateway" "nat" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "${local.name}-nat"
}

# Public RT (IPv4+IPv6 to IGW)
resource "oci_core_route_table" "rt_public" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "${local.name}-rt-public"

  route_rules {
    destination       = "0.0.0.0/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_internet_gateway.igw.id
  }

  route_rules {
    destination       = "::/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_internet_gateway.igw.id
  }
}

# Private RT (IPv4 to NAT only; NO ::/0)
resource "oci_core_route_table" "rt_private" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "${local.name}-rt-private"

  route_rules {
    destination       = "0.0.0.0/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_nat_gateway.nat.id
  }
}

# Default Security List — TEST MODEL (NOT PROD): protocol "all" stateless egress+ingress (v4+v6)
resource "oci_core_default_security_list" "vcn_default" {
  manage_default_resource_id = oci_core_vcn.vcn.default_security_list_id

  # Egress all v4
  egress_security_rules {
    protocol         = "all"
    destination      = "0.0.0.0/0"
    destination_type = "CIDR_BLOCK"
    stateless        = true
  }

  # Egress all v6
  egress_security_rules {
    protocol         = "all"
    destination      = "::/0"
    destination_type = "CIDR_BLOCK"
    stateless        = true
  }

  # Ingress all v4
  ingress_security_rules {
    protocol    = "all"
    source      = "0.0.0.0/0"
    source_type = "CIDR_BLOCK"
    stateless   = true
  }

  # Ingress all v6
  ingress_security_rules {
    protocol    = "all"
    source      = "::/0"
    source_type = "CIDR_BLOCK"
    stateless   = true
  }
}

# Derive IPv6 subnets from the VCN base
locals {
  vcn_ipv6_base   = tolist(oci_core_vcn.vcn.ipv6cidr_blocks)[0]
  cp_ipv6_cidr    = cidrsubnet(local.vcn_ipv6_base, 8, 1)
  plb_ipv6_cidr   = cidrsubnet(local.vcn_ipv6_base, 8, 2)
  wprv_ipv6_cidr  = cidrsubnet(local.vcn_ipv6_base, 8, 3)
  pods_ipv6_cidr  = cidrsubnet(local.vcn_ipv6_base, 8, 4)
  gens_ipv6_cidr  = cidrsubnet(local.vcn_ipv6_base, 8, 5)
}

# Subnets
resource "oci_core_subnet" "cp" {
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "${local.name}-cp"
  cidr_block                 = var.cp_cidr
  ipv6cidr_block             = local.cp_ipv6_cidr
  prohibit_public_ip_on_vnic = false
  route_table_id             = oci_core_route_table.rt_public.id
  dns_label                  = "cp"
}

resource "oci_core_subnet" "pub_lb" {
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "${local.name}-pub-lb"
  cidr_block                 = var.pub_lb_cidr
  ipv6cidr_block             = local.plb_ipv6_cidr
  prohibit_public_ip_on_vnic = false
  route_table_id             = oci_core_route_table.rt_public.id
  dns_label                  = "publb"
}

resource "oci_core_subnet" "workers_private" {
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "${local.name}-workers-prv"
  cidr_block                 = var.workers_private_cidr
  ipv6cidr_block             = local.wprv_ipv6_cidr
  prohibit_public_ip_on_vnic = true
  route_table_id             = oci_core_route_table.rt_private.id
  dns_label                  = "wksprv"
}

resource "oci_core_subnet" "pods" {
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "${local.name}-pods"
  cidr_block                 = var.pods_cidr
  ipv6cidr_block             = local.pods_ipv6_cidr
  prohibit_public_ip_on_vnic = true
  route_table_id             = oci_core_route_table.rt_private.id
  dns_label                  = "pods"
}

resource "oci_core_subnet" "gens_pub" {
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "${local.name}-gens-pub"
  cidr_block                 = var.gens_pub_cidr
  ipv6cidr_block             = local.gens_ipv6_cidr
  prohibit_public_ip_on_vnic = false
  route_table_id             = oci_core_route_table.rt_public.id
  dns_label                  = "genspub"
}

# NSGs — TEST MODEL (NOT PROD): "all/all stateless" in both directions and both families

resource "oci_core_network_security_group" "nsg_lb" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "${local.name}-nsg-lb"
}

resource "oci_core_network_security_group_security_rule" "nsg_lb_egress_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "0.0.0.0/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "nsg_lb_egress_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "::/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "nsg_lb_ingress_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "INGRESS"
  protocol                  = "all"
  source_type               = "CIDR_BLOCK"
  source                    = "0.0.0.0/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "nsg_lb_ingress_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "INGRESS"
  protocol                  = "all"
  source_type               = "CIDR_BLOCK"
  source                    = "::/0"
  stateless                 = true
}

resource "oci_core_network_security_group" "nsg_backends" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "${local.name}-nsg-backends"
}

resource "oci_core_network_security_group_security_rule" "nsg_back_egress_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "0.0.0.0/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "nsg_back_egress_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "::/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "nsg_back_ingress_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "INGRESS"
  protocol                  = "all"
  source_type               = "CIDR_BLOCK"
  source                    = "0.0.0.0/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "nsg_back_ingress_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "INGRESS"
  protocol                  = "all"
  source_type               = "CIDR_BLOCK"
  source                    = "::/0"
  stateless                 = true
}

resource "oci_core_network_security_group" "nsg_generators" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "${local.name}-nsg-generators"
}

resource "oci_core_network_security_group_security_rule" "gens_egress_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "0.0.0.0/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "gens_egress_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "::/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "gens_ingress_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "all"
  source_type               = "CIDR_BLOCK"
  source                    = "0.0.0.0/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "gens_ingress_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "all"
  source_type               = "CIDR_BLOCK"
  source                    = "::/0"
  stateless                 = true
}
""").strip("\n")

oke_tf = textwrap.dedent("""
# OKE cluster and backend node pool (conditional via count)
resource "oci_containerengine_cluster" "this" {
  count              = var.enable_oke ? 1 : 0
  compartment_id     = var.compartment_id
  name               = local.name
  kubernetes_version = var.kubernetes_version
  vcn_id             = oci_core_vcn.vcn.id

  # Native CNI (TOP-LEVEL)
  cluster_pod_network_options {
    cni_type = "OCI_VCN_IP_NATIVE"
  }

  endpoint_config {
    is_public_ip_enabled = true
    subnet_id            = oci_core_subnet.cp.id
  }

  options {
    # service LB on public subnet; IPv4 only for stability
    service_lb_subnet_ids = [ oci_core_subnet.pub_lb.id ]
    ip_families = ["IPv4"]
  }
}

resource "oci_containerengine_node_pool" "backend" {
  count              = var.enable_oke ? 1 : 0
  compartment_id     = var.compartment_id
  cluster_id         = oci_containerengine_cluster.this[0].id
  kubernetes_version = var.kubernetes_version
  name               = "${local.name}-backend"
  node_shape         = var.backend_node_shape
  ssh_public_key     = file(var.ssh_public_key_path_oke)

  node_config_details {
    size = var.backend_pool_size

    placement_configs {
      availability_domain = var.ad_a
      subnet_id           = oci_core_subnet.workers_private.id
    }

    node_pool_pod_network_option_details {
      cni_type          = "OCI_VCN_IP_NATIVE"
      max_pods_per_node = var.max_pods_per_node
      pod_nsg_ids       = []
      pod_subnet_ids    = [ oci_core_subnet.pods.id ]
    }
  }

  # Flex shape config if applicable (safe across fixed shapes; ignored when not Flex)
  node_shape_config {
    ocpus         = var.backend_ocpus
    memory_in_gbs = var.backend_memory_gb
  }

  node_source_details {
    source_type = "IMAGE"
    image_id    = var.node_image_ocid_backend
  }
}
""").strip("\n")

generators_tf = textwrap.dedent("""
# Generator instances (conditional by enable_generators)
resource "oci_core_instance" "generator" {
  count               = var.enable_generators ? var.generator_count : 0
  availability_domain = var.ad_a
  compartment_id      = var.compartment_id
  shape               = var.generator_shape

  # Flex shape config if applicable
  dynamic "shape_config" {
    for_each = endswith(var.generator_shape, ".Flex") ? [1] : []
    content {
      ocpus         = var.generator_ocpus
      memory_in_gbs = var.generator_memory_gb
    }
  }

  source_details {
    source_type = "image"
    source_id   = var.generator_image_id
  }

  create_vnic_details {
    subnet_id        = oci_core_subnet.gens_pub.id
    assign_public_ip = true
    nsg_ids          = [ oci_core_network_security_group.nsg_generators.id ]
  }

  agent_config {
    are_all_plugins_disabled = false
    is_management_disabled   = false
    is_monitoring_disabled   = false

    plugins_config {
      name          = var.bastion_plugin_name
      desired_state = "ENABLED"
    }
  }

  display_name = "ext-gen-${count.index}"

  metadata = {
    ssh_authorized_keys = var.ssh_public_key_content_gen
    user_data           = filebase64("${path.module}/cloud-init/generator.sh")
  }
}

# Optional IPv6 assignment per generator (only if enabled)
data "oci_core_vnic_attachments" "gen_vnic" {
  count          = (var.enable_generators && var.enable_ipv6_on_generators) ? var.generator_count : 0
  compartment_id = var.compartment_id
  instance_id    = oci_core_instance.generator[count.index].id
}

resource "oci_core_ipv6" "gen_v6" {
  count     = (var.enable_generators && var.enable_ipv6_on_generators) ? var.generator_count : 0
  subnet_id = oci_core_subnet.gens_pub.id
  vnic_id   = data.oci_core_vnic_attachments.gen_vnic[count.index].vnic_attachments[0].vnic_id
}
""").strip("\n")

outputs_tf = textwrap.dedent("""
output "vcn_id" {
  value = oci_core_vcn.vcn.id
}

output "subnets" {
  value = {
    control_plane   = oci_core_subnet.cp.id
    public_lb       = oci_core_subnet.pub_lb.id
    workers_private = oci_core_subnet.workers_private.id
    pods            = oci_core_subnet.pods.id
    gens_pub        = oci_core_subnet.gens_pub.id
  }
}

output "cluster_id" {
  value = var.enable_oke ? oci_containerengine_cluster.this[0].id : null
}

output "gen_public_ips" {
  value = var.enable_generators ? [for i in oci_core_instance.generator : i.public_ip] : []
}

output "gen_ipv6_ips" {
  value = var.enable_generators ? oci_core_ipv6.gen_v6[*].ip_address : []
}
""").strip("\n")

# --------- cloud-init: generator.sh (tuning-only; includes locustfile.py) ---------

generator_cloud_init = r"""#!/bin/bash
set -euo pipefail

# Stop/disable firewalld if present
if command -v firewall-cmd >/dev/null 2>&1; then
  systemctl stop firewalld || true
  systemctl disable firewalld || true
fi

# Enable Oracle Cloud Agent if present
if systemctl list-unit-files | grep -q oracle-cloud-agent.service; then
  systemctl enable --now oracle-cloud-agent || true
fi

# Kernel tuning (safe subset)
cat <<'EOF' >/etc/sysctl.d/99-cps.conf
net.core.somaxconn=65535
net.core.netdev_max_backlog=250000
net.ipv4.tcp_max_syn_backlog=262144
net.ipv4.ip_local_port_range=1024 65535
net.ipv4.tcp_fin_timeout=15
fs.file-max=1000000
EOF
sysctl --system || true

# Raise ulimit
echo "* - nofile 1048576" >> /etc/security/limits.conf || true

# Workspace
mkdir -p /home/opc/locustwork/results
chown -R opc:opc /home/opc/locustwork

# Multi-capable locustfile (targets & behavior driven by environment)
cat > /home/opc/locustwork/locustfile.py <<'PY'
import os
from itertools import cycle
from locust import HttpUser, task, constant

VERIFY_TLS        = os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"
CONNECT_TIMEOUT_S = float(os.environ.get("LOCUST_CONNECT_TIMEOUT_S", "8"))
READ_TIMEOUT_S    = float(os.environ.get("LOCUST_READ_TIMEOUT_S", "15"))
WAIT_TIME_S       = float(os.environ.get("LOCUST_WAIT_TIME_S", "1.0"))
MODE              = os.environ.get("LOCUST_MODE", "cps").lower()
HEALTH_PATH       = os.environ.get("LOCUST_HEALTH_PATH", "/healthz")
THROUGHPUT_PATH   = os.environ.get("LOCUST_THROUGHPUT_PATH", "/payload_100k")
RAW_TARGETS       = os.environ.get("LOCUST_TARGETS", "")
TARGETS           = [t.strip() for t in RAW_TARGETS.split(",") if t.strip()]
NAME_BY_VIP       = os.environ.get("LOCUST_NAME_BY_VIP", "true").lower() == "true"

class CpsUser(HttpUser):
    host = os.environ.get("LOCUST_DEFAULT_HOST", "")
    wait_time = constant(WAIT_TIME_S)

    def on_start(self):
        self._targets = cycle(TARGETS) if TARGETS else None

    def _next_base(self):
        try:
            return next(self._targets) if self._targets else (self.host or "")
        except Exception:
            return self.host or ""

    @task
    def do_request(self):
        path = HEALTH_PATH if MODE == "cps" else THROUGHPUT_PATH
        base = self._next_base()
        url  = f"{base}{path}" if base else path
        headers = {"Connection": "close"} if MODE == "cps" else {}
        name_val = path
        if NAME_BY_VIP and base:
            vip = base.replace("https://","").replace("http://","")
            name_val = f"{vip}{path}"
        self.client.get(url,
                        headers=headers,
                        verify=VERIFY_TLS,
                        timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
                        name=name_val)
PY

chown -R opc:opc /home/opc/locustwork
echo READY
"""

# --------- terraform.tfvars from environment ---------

def _to_int(v: str, default: int) -> int:
    try:
        return int(float(v))
    except Exception:
        return default

tfvars = textwrap.dedent(f"""
oci_profile       = "{OCI_PROFILE}"
region            = "{REGION}"
compartment_id    = "{COMPARTMENT_ID}"
ad_a              = "{AD_A}"

enable_oke        = {"true" if ENABLE_OKE else "false"}
enable_generators = {"true" if ENABLE_GENERATORS else "false"}

cp_cidr              = "{CP_CIDR}"
pub_lb_cidr          = "{PUB_LB_CIDR}"
workers_private_cidr = "{WORKERS_PRIVATE_CIDR}"
pods_cidr            = "{PODS_CIDR}"
gens_pub_cidr        = "{GENS_PUB_CIDR}"

kubernetes_version      = "{KUBERNETES_VERSION}"
ssh_public_key_path_oke = "{SSH_PUBLIC_KEY_PATH_OKE}"
backend_node_shape      = "{BACKEND_NODE_SHAPE}"
backend_ocpus           = {_to_int(BACKEND_OCPUS, 16)}
backend_memory_gb       = {_to_int(BACKEND_MEMORY_GB, 64)}
max_pods_per_node       = {_to_int(MAX_PODS_PER_NODE, 31)}
backend_pool_size       = {_to_int(BACKEND_POOL_SIZE, 1)}
node_image_ocid_backend = "{NODE_IMAGE_OCID_BACKEND}"

generator_count        = {_to_int(GENERATOR_COUNT, 1)}
generator_shape        = "{GENERATOR_SHAPE}"
generator_ocpus        = {_to_int(GENERATOR_OCPUS, 8)}
generator_memory_gb    = {_to_int(GENERATOR_MEMORY_GB, 32)}
enable_ipv6_on_generators = {"true" if ENABLE_IPV6_ON_GENERATORS else "false"}
generator_image_id     = "{GEN_IMAGE_ID}"
ssh_public_key_content_gen = "{(GEN_SSH_PUBLIC_KEY_CONTENT or '').replace('"','\\"')}"
ssh_private_key_path_gen   = "{GEN_SSH_PRIVATE_KEY_PATH}"
""").strip("\n")

# --------- Write files ---------

files = {
    stack_dir / "provider.tf": provider_tf,
    stack_dir / "variables.tf": variables_tf,
    stack_dir / "locals.tf": locals_tf,
    stack_dir / "network.tf": network_tf,
    stack_dir / "oke.tf": oke_tf,
    stack_dir / "generators.tf": generators_tf,
    stack_dir / "outputs.tf": outputs_tf,
    stack_dir / "terraform.tfvars": tfvars,
    cloud_init_dir / "generator.sh": generator_cloud_init,
}

for path, content in files.items():
    path.write_text(content, encoding="utf-8")

print(f"Cell 4 complete: Wrote Terraform stack and cloud-init under: {stack_dir.resolve()}")
print("Files:")
for p in files.keys():
    print(" -", p)

print("\nNotes:")
print(" - COMPLETE STATELESS posture: Default Security List and all NSGs include symmetric protocol=all stateless rules for IPv4 and IPv6 (egress+ingress).")
print(" - Private subnets (workers_private, pods) use NAT for IPv4 only (no ::/0); public subnets route v4+v6 via IGW.")
print(" - OKE and Generators are conditionally created via enable_oke / enable_generators (from DEPLOY_MODE).")
print(" - Generator instances use generator_image_id from Cell 3 (shape-compatible Oracle Linux image).")
print(" - Next: Run Cell 5 to terraform init/plan/apply and capture outputs.")


In [ ]:
# Cell 5 — Objective: Terraform init/validate/plan/apply for ./stack + capture outputs
# - Honors RUN_INFRA_APPLY (skips if false)
# - Optional pre-destroy if any DESTROY_*_ON_APPLY flags are true (full destroy, then fresh apply)
# - Cleans .terraform/ and lock file to avoid stale state
# - Saves terraform outputs to stack-outputs.json and a timestamped copy under OUTPUT_DIR
# - Persists GEN_IPS_V4_JSON and GEN_IPS_V6_JSON to the environment for later cells

import os, json, subprocess, shlex, time
from pathlib import Path
from datetime import datetime, timezone

def _fail(msg: str):
    raise RuntimeError(msg)

def _run(cmd: str, cwd: Path | None = None, check: bool = True) -> int:
    print(f"\n$ {cmd}")
    rc = subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None).returncode
    if check and rc != 0:
        _fail(f"Command failed ({rc}): {cmd}")
    return rc

def _run_capture(cmd: str, cwd: Path | None = None, check: bool = True) -> tuple[int, str, str]:
    print(f"\n$ {cmd}")
    p = subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if check and p.returncode != 0:
        print((p.stdout or "").strip())
        print((p.stderr or "").strip())
        _fail(f"Command failed ({p.returncode}): {cmd}")
    return p.returncode, (p.stdout or ""), (p.stderr or "")

# Toggles and context from Cell 2/3
RUN_INFRA_APPLY       = os.environ.get("RUN_INFRA_APPLY", "true").strip().lower() in ("1","true","yes","y")
DESTROY_GENS_ON_APPLY = os.environ.get("DESTROY_GENS_ON_APPLY", "false").strip().lower() in ("1","true","yes","y")
DESTROY_OKE_ON_APPLY  = os.environ.get("DESTROY_OKE_ON_APPLY",  "false").strip().lower() in ("1","true","yes","y")
DESTROY_NETWORK_ON_APPLY = os.environ.get("DESTROY_NETWORK_ON_APPLY", "false").strip().lower() in ("1","true","yes","y")

OUTPUT_DIR = Path(os.path.abspath(os.environ.get("OUTPUT_DIR", "./results")))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TS_UTC = os.environ.get("TS_UTC", "") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
os.environ["TS_UTC"] = TS_UTC

# Workdir
stack_dir = Path("stack").resolve()
if not stack_dir.is_dir():
    _fail(f"Terraform working directory not found: {stack_dir}")

# Required files sanity
required = [
    "provider.tf", "variables.tf", "locals.tf",
    "network.tf", "oke.tf", "generators.tf", "outputs.tf",
    "terraform.tfvars"
]
missing = [f for f in required if not (stack_dir / f).exists()]
if missing:
    _fail(f"Missing Terraform files in {stack_dir}: {missing}")

# Optional: quick check for terraform binary (informative)
try:
    import shutil
    if shutil.which("terraform") is None:
        print("Warning: 'terraform' not found in PATH. Install Terraform and re-run this cell.")
except Exception:
    pass

if not RUN_INFRA_APPLY:
    print("RUN_INFRA_APPLY is false — skipping Terraform init/plan/apply.")
else:
    # If any DESTROY_*_ON_APPLY flags are set, perform a full destroy before apply
    if DESTROY_GENS_ON_APPLY or DESTROY_OKE_ON_APPLY or DESTROY_NETWORK_ON_APPLY:
        print("One or more DESTROY_*_ON_APPLY flags are true — performing a full terraform destroy before re-apply.")
        _run("terraform init -upgrade -input=false", cwd=stack_dir, check=True)
        _run("terraform destroy -parallelism=20 -auto-approve -var-file=terraform.tfvars", cwd=stack_dir, check=True)

    # Clean artifacts to avoid stale state
    lockfile = stack_dir / ".terraform.lock.hcl"
    tf_dir   = stack_dir / ".terraform"
    if lockfile.exists():
        print(f"Removing {lockfile}")
        try: lockfile.unlink()
        except Exception as e:
            print(f"Note: could not remove lockfile: {e}")
    if tf_dir.exists() and tf_dir.is_dir():
        print(f"Removing {tf_dir}")
        _run(f"rm -rf {shlex.quote(str(tf_dir))}", cwd=stack_dir, check=True)

    # Init with retry
    rc = _run("terraform init -upgrade -input=false", cwd=stack_dir, check=False)
    if rc != 0:
        print("Init failed; retrying in 5s ...")
        time.sleep(5)
        _run("terraform init -upgrade -input=false", cwd=stack_dir, check=True)

    # Validate
    _run("terraform validate", cwd=stack_dir, check=True)

    # Plan
    tfplan = "tfplan"
    _run(f"terraform plan -input=false -out={shlex.quote(tfplan)}", cwd=stack_dir, check=True)

    # Optional: preview first ~200 lines of plan
    print("\n--- Plan preview (first ~200 lines) ---")
    _run(f"terraform show -no-color {shlex.quote(tfplan)} | sed -n '1,200p'", cwd=stack_dir, check=False)

    # Apply
    _run(f"terraform apply -input=false -auto-approve {shlex.quote(tfplan)}", cwd=stack_dir, check=True)

    # Outputs to JSON + summary
    outputs_json = stack_dir / "stack-outputs.json"
    _run(f"terraform output -json > {shlex.quote(str(outputs_json))}", cwd=stack_dir, check=True)

    # Parse outputs
    outputs = {}
    try:
        outputs = json.loads(outputs_json.read_text(encoding="utf-8") or "{}")
    except Exception as e:
        print(f"Note: Could not parse outputs JSON. Reason: {e!r}")

    def _val(obj: dict, key: str):
        item = obj.get(key, {})
        if isinstance(item, dict) and "value" in item:
            return item.get("value")
        return item if item is not None else None

    VCN_ID = _val(outputs, "vcn_id")
    SUBNETS = _val(outputs, "subnets") or {}
    CLUSTER_ID = _val(outputs, "cluster_id")
    GEN_V4 = _val(outputs, "gen_public_ips") or []
    GEN_V6 = _val(outputs, "gen_ipv6_ips") or []

    # Persist to environment for downstream cells
    os.environ["GEN_IPS_V4_JSON"] = json.dumps(GEN_V4)
    os.environ["GEN_IPS_V6_JSON"] = json.dumps(GEN_V6)

    # Save a timestamped copy of outputs under OUTPUT_DIR
    ts_copy = OUTPUT_DIR / f"stack_outputs_{TS_UTC}.json"
    try:
        ts_copy.write_text(json.dumps({
            "vcn_id": VCN_ID,
            "subnets": SUBNETS,
            "cluster_id": CLUSTER_ID,
            "gen_public_ips": GEN_V4,
            "gen_ipv6_ips": GEN_V6,
            "ts_utc": TS_UTC
        }, indent=2), encoding="utf-8")
    except Exception as e:
        print(f"Note: Could not write timestamped outputs file. Reason: {e!r}")

    # Summary
    print("\n--- Terraform outputs (summary) ---")
    print("vcn_id:", VCN_ID)
    print("subnets:", SUBNETS)
    print("cluster_id:", CLUSTER_ID)
    print("gen_public_ips (IPv4):", GEN_V4)
    print("gen_ipv6_ips (IPv6):", GEN_V6)
    print(f"\nFull outputs saved to: {outputs_json}")
    if ts_copy.exists():
        print(f"Timestamped copy saved to: {ts_copy}")

print("\nCell 5 complete.")
print("NEXT:")
print(" - If ENABLE_OKE and RUN_K8S_DEPLOY=true: proceed to Cell 6 (kubeconfig/namespace/TLS secret).")
print(" - If ENABLE_GENERATORS and RUN_GENERATORS_PREP=true: after Cell 6/7, you will run Cell 9 to prepare generators.")
print(" - With RUN_LOCUST=true and no explicit targets, Cell 10 will auto-target the discovered OKE VIP when available.")


In [ ]:
# Cell 6 — Objective: Configure kubeconfig (if possible), create namespace, and provision TLS secret for OKE Service
# - Reads cluster_id from ./stack/stack-outputs.json
# - If 'oci' CLI is available: fetch kubeconfig for the OKE cluster (public endpoint)
# - Verifies kubectl context with version calls that avoid unsupported flags
# - Creates/ensures namespace (idempotent)
# - Generates a self‑signed TLS cert/key (RSA-2048 by default) and (re)creates secret 'ssl-certificate-secret'
# - Honors RUN_K8S_DEPLOY and ENABLE_OKE; skips with reason if disabled

import os, subprocess, shlex, json
from pathlib import Path

def _run(cmd: str, check: bool = True, cwd: Path | None = None) -> tuple[int, str, str]:
    print(f"\n$ {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, cwd=str(cwd) if cwd else None,
                       capture_output=True)
    out, err = (p.stdout or "").strip(), (p.stderr or "").strip()
    if out:
        print(out)
    # Print stderr on error/fail text to aid troubleshooting
    if err and (("error" in err.lower()) or ("failed" in err.lower())):
        print(err)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}\n{out}\n{err}")
    return p.returncode, out, err

def _which(name: str) -> bool:
    try:
        import shutil
        return shutil.which(name) is not None
    except Exception:
        return False

def _load_stack_outputs() -> dict:
    p = Path("stack/stack-outputs.json").resolve()
    if not p.exists():
        raise FileNotFoundError(f"stack-outputs.json not found at {p}")
    try:
        return json.loads(p.read_text(encoding="utf-8") or "{}")
    except json.JSONDecodeError as e:
        raise RuntimeError(f"Could not parse stack-outputs.json: {e}")

# Toggles and inputs from prior cells
RUN_K8S_DEPLOY  = os.environ.get("RUN_K8S_DEPLOY", "true").strip().lower() in ("1","true","yes","y")
DEPLOY_MODE     = os.environ.get("DEPLOY_MODE", "both").strip().lower()
ENABLE_OKE      = (DEPLOY_MODE in ("oke-only", "both"))
WORK_NS         = os.environ.get("WORK_NS", "lbtest").strip() or "lbtest"
REGION          = os.environ.get("REGION", "").strip()

# TLS selection (from Cell 2; forced RSA unless changed there)
TLS_KEY_ALGO    = os.environ.get("TLS_KEY_ALGO", "rsa").strip().lower()       # "rsa" | "ecdsa"
TLS_ECDSA_CURVE = os.environ.get("TLS_ECDSA_CURVE", "prime256v1").strip()     # used only if TLS_KEY_ALGO=ecdsa

if not RUN_K8S_DEPLOY:
    print("RUN_K8S_DEPLOY is false — skipping kubeconfig/namespace/TLS secret setup.")
elif not ENABLE_OKE:
    print("ENABLE_OKE is false (DEPLOY_MODE != oke-only/both) — skipping kubeconfig/namespace/TLS secret setup.")
else:
    # Load cluster_id from Terraform outputs
    outs = _load_stack_outputs()
    cluster_id = (outs.get("cluster_id") or {}).get("value") if isinstance(outs.get("cluster_id"), dict) else outs.get("cluster_id")
    if not cluster_id:
        raise RuntimeError("cluster_id not found in stack-outputs.json. Ensure Cell 5 completed successfully.")
    print(f"Cluster ID: {cluster_id}")

    # 1) Kubeconfig setup if 'oci' CLI is present; otherwise print instructions
    if not REGION:
        raise RuntimeError("REGION is empty in environment. Ensure Cell 3 Apply persisted REGION.")
    if _which("oci"):
        home = Path.home()
        kube_dir = home / ".kube"
        kube_dir.mkdir(parents=True, exist_ok=True)
        kubeconfig_path = kube_dir / "config"
        print(f"Attempting to write kubeconfig to: {kubeconfig_path}")
        _run(
            f"oci ce cluster create-kubeconfig "
            f"--cluster-id {shlex.quote(cluster_id)} "
            f"--file {shlex.quote(str(kubeconfig_path))} "
            f"--region {shlex.quote(REGION)} "
            f"--token-version 2.0.0 "
            f"--kube-endpoint PUBLIC_ENDPOINT "
            f"--overwrite",
            check=True
        )
        _run("kubectl version --client", check=False)
        _run("kubectl version", check=False)
        _run("kubectl config current-context", check=False)
    else:
        print("oci CLI not found — Skipping automatic kubeconfig setup.")
        print("Manual option (from your shell):")
        print(f"  oci ce cluster create-kubeconfig --cluster-id {cluster_id} "
              f"--file ~/.kube/config --region {REGION} --token-version 2.0.0 "
              f"--kube-endpoint PUBLIC_ENDPOINT --overwrite")
        print("After that, verify with:")
        print("  kubectl version --client")
        print("  kubectl version")
        print("  kubectl config current-context")

    # 2) Create namespace (idempotent)
    print(f"\nEnsuring namespace exists: {WORK_NS}")
    rc, out, err = _run(f"kubectl get ns {shlex.quote(WORK_NS)}", check=False)
    if rc != 0:
        _run(f"kubectl create namespace {shlex.quote(WORK_NS)}", check=True)
    else:
        print(f"Namespace {WORK_NS} already exists.")

    # 3) Generate TLS cert/key (RSA-2048 by default) and (re)create secret
    tls_key = Path("tls.key").resolve()
    tls_crt = Path("tls.crt").resolve()

    if TLS_KEY_ALGO == "ecdsa":
        print(f"\nGenerating self-signed TLS certificate (ECDSA curve={TLS_ECDSA_CURVE}, 365 days) ...")
        _run(f"openssl ecparam -name {shlex.quote(TLS_ECDSA_CURVE)} -genkey -noout -out {shlex.quote(str(tls_key))}", check=True)
        _run(
            "openssl req -x509 -new "
            f"-key {shlex.quote(str(tls_key))} "
            "-sha256 -days 365 "
            "-subj \"/CN=nginxsvc/O=nginxsvc\" "
            f"-out {shlex.quote(str(tls_crt))}",
            check=True
        )
    else:
        print("\nGenerating self-signed TLS certificate (RSA:2048, 365 days) ...")
        _run(
            "openssl req -x509 -nodes -days 365 -newkey rsa:2048 "
            f"-keyout {shlex.quote(str(tls_key))} "
            f"-out {shlex.quote(str(tls_crt))} "
            "-subj \"/CN=nginxsvc/O=nginxsvc\"",
            check=True
        )

    # Optional: verify cert key type (informational)
    _run(f"openssl x509 -in {shlex.quote(str(tls_crt))} -noout -text | grep -E 'Public Key Algorithm|Curve|RSA Public-Key' || true", check=False)

    secret_name = "ssl-certificate-secret"
    print(f"\n(Re)creating TLS secret '{secret_name}' in namespace '{WORK_NS}' ...")
    _run(f"kubectl -n {shlex.quote(WORK_NS)} delete secret {shlex.quote(secret_name)} --ignore-not-found", check=False)
    _run(
        f"kubectl -n {shlex.quote(WORK_NS)} create secret tls {shlex.quote(secret_name)} "
        f"--key {shlex.quote(str(tls_key))} --cert {shlex.quote(str(tls_crt))}",
        check=True
    )
    _run(f"kubectl -n {shlex.quote(WORK_NS)} get secret {shlex.quote(secret_name)} -o yaml | head -n 20", check=False)

    # 4) Summary and next steps
    print("\nCell 6 complete:")
    print(f" - kubeconfig: {'configured via oci CLI' if _which('oci') else 'not configured (instructions printed)'}")
    print(f" - namespace: {WORK_NS} present")
    print(f" - secret: {secret_name} created in ns/{WORK_NS} (key_algo={TLS_KEY_ALGO})")
    print("\nNEXT:")
    print(" - Run Cell 7 to (re)deploy ds-backend-tuner, NGINX DaemonSet, and the LoadBalancer Service;")
    print("   it performs an immutable-safe Service recreate so the OCI LB picks up the RSA certificate.")


In [ ]:
# Cell 7 — Objective: Deploy backend tuner + NGINX DaemonSet (hostNetwork) + LoadBalancer Service in WORK_NS, then validate VIP
# - Tuner DS: applies Stage D sysctls; waits Ready + 60s settle
# - ConfigMap: nginx.conf (combined-aligned safe subset; reuseport; backlog=262144; PPv2)
# - NGINX DS: hostNetwork: true, dnsPolicy: ClusterFirstWithHostNet, hostPort: 30080 (and containerPort: 30080), nodeSelector app.role=backend
# - Service: type LoadBalancer, externalTrafficPolicy: Local, targetPort: 30080 (let nodePort auto-assign), TLS offload on 443, PPv2=2, flexible shape 8000/8000
# - Wait DS rollout; discover EXTERNAL-IP/hostname; sleep 30s; validate HEAD /healthz
# - Honors RUN_K8S_DEPLOY and WORK_NS; warns if no backend-labeled nodes

import os, json, shlex, time, subprocess, textwrap
from pathlib import Path

def _run(cmd: str, check: bool = False, quiet: bool = False):
    if not quiet:
        print(f"\n$ {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if not quiet:
        if p.stdout: print(p.stdout.strip())
        if p.stderr and (("error" in p.stderr.lower()) or ("failed" in p.stderr.lower())):
            print(p.stderr.strip())
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}\n{p.stdout}\n{p.stderr}")
    return p

def _jsonpath_svc(ns: str, expr: str) -> str:
    q = f"kubectl -n {shlex.quote(ns)} get svc nginx-service -o jsonpath='{expr}'"
    p = subprocess.run(["bash","-lc", q], capture_output=True, text=True)
    return (p.stdout or "").strip()

RUN_K8S_DEPLOY = os.environ.get("RUN_K8S_DEPLOY", "true").strip().lower() in ("1","true","yes","y")
WORK_NS = os.environ.get("WORK_NS", "lbtest").strip() or "lbtest"

if not RUN_K8S_DEPLOY:
    print("RUN_K8S_DEPLOY is false — skipping backend deployment.")
else:
    # Pre-flight: backend-labeled nodes?
    p_nodes = _run("kubectl get nodes -l app.role=backend -o name", quiet=True)
    if not (p_nodes.stdout or "").strip():
        print("WARNING: No nodes labeled app.role=backend. Label at least one node for NGINX DS scheduling:")
        print("  kubectl get nodes -o wide")
        print("  kubectl label node <BACKEND_NODE_NAME> app.role=backend --overwrite")

    # Pre-flight: TLS secret present?
    p_tls = _run(f"kubectl -n {shlex.quote(WORK_NS)} get secret ssl-certificate-secret -o name", quiet=True)
    if not (p_tls.stdout or "").strip():
        print(f"WARNING: TLS secret 'ssl-certificate-secret' not found in ns/{WORK_NS}. Re-run Cell 6 before this cell.")

    # 1) Backend tuner (Stage D)
    tuner_yaml = textwrap.dedent(f"""
    apiVersion: apps/v1
    kind: DaemonSet
    metadata:
      name: ds-backend-tuner
      namespace: {WORK_NS}
    spec:
      selector:
        matchLabels:
          app: backend-tuner
          stage: final
      template:
        metadata:
          labels:
            app: backend-tuner
            stage: final
        spec:
          hostNetwork: true
          hostPID: true
          serviceAccountName: default
          nodeSelector:
            app.role: backend
          tolerations:
          - operator: "Exists"
          containers:
          - name: tuner
            image: oraclelinux:8-slim
            imagePullPolicy: IfNotPresent
            securityContext:
              privileged: true
              allowPrivilegeEscalation: true
              readOnlyRootFilesystem: false
            resources:
              requests: {{ cpu: "10m", memory: "32Mi" }}
              limits:   {{ cpu: "100m", memory: "128Mi" }}
            command: ["/bin/sh","-c"]
            args:
            - |
              set -euo pipefail
              SYSCTL_BIN="/sbin/sysctl"
              if ! nsenter -t 1 -m -u -i -n -p -- test -x "$SYSCTL_BIN"; then
                if nsenter -t 1 -m -u -i -n -p -- test -x /usr/sbin/sysctl; then
                  SYSCTL_BIN="/usr/sbin/sysctl"
                else
                  SYSCTL_BIN="sysctl"
                fi
              fi
              nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -w net.core.somaxconn=262144
              nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -w net.core.netdev_max_backlog=500000
              nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -w net.ipv4.tcp_max_syn_backlog=262144
              nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -w net.ipv4.ip_local_port_range="1024 65535"
              nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -w net.ipv4.tcp_fin_timeout=15
              echo "Host values after apply:"
              nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -n net.core.somaxconn
              nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -n net.core.netdev_max_backlog
              nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -n net.ipv4.tcp_max_syn_backlog
              nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -n net.ipv4.ip_local_port_range
              nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -n net.ipv4.tcp_fin_timeout
              echo "Final Stage D sysctls applied."
              sleep infinity
            volumeMounts:
            - name: nsenter
              mountPath: /usr/bin/nsenter
              readOnly: true
          volumes:
          - name: nsenter
            hostPath: {{ path: /usr/bin/nsenter, type: File }}
    """).strip("\n")

    Path("ds-backend-tuner.yaml").write_text(tuner_yaml, encoding="utf-8")
    print(f"Applying ds-backend-tuner in ns/{WORK_NS} ...")
    _run(f"kubectl apply -f ds-backend-tuner.yaml", check=True)

    # Wait for tuner DS Ready (up to 2 min)
    print("\nWaiting for ds-backend-tuner to be Ready (up to 120s) ...")
    deadline = time.time() + 120
    ds_ok = False
    while time.time() < deadline and not ds_ok:
        p = _run(f"kubectl -n {shlex.quote(WORK_NS)} get ds ds-backend-tuner -o json", quiet=True)
        try:
            ds = json.loads(p.stdout or "{}")
            status = ds.get("status") or {}
            desired = int(status.get("desiredNumberScheduled") or 0)
            ready_ct = int(status.get("numberReady") or 0)
            print(f"Desired={desired} Ready={ready_ct}")
            ds_ok = (desired > 0 and ready_ct >= desired)
        except Exception:
            pass
        if not ds_ok:
            time.sleep(4)
    _run(f"kubectl -n {shlex.quote(WORK_NS)} get ds ds-backend-tuner -o wide", check=False)

    # Settle nodes (no traffic yet)
    print("\nAllowing node settle time (sleep 60s) ...")
    time.sleep(60)

    # 2) NGINX ConfigMap — build robust YAML with properly indented block scalar
    nginx_conf = """user  nginx;
worker_processes  auto;
worker_rlimit_nofile 1048576;

events {
    worker_connections  131072;
    multi_accept on;
    accept_mutex off;
}

http {
    include       /etc/nginx/mime.types;
    default_type  application/octet-stream;

    # I/O and TCP toggles
    sendfile on;
    tcp_nopush on;
    tcp_nodelay on;

    # CPS-friendly timeouts
    keepalive_timeout 15;
    client_body_timeout 10;
    client_header_timeout 10;
    send_timeout 10;
    reset_timedout_connection on;

    # Reduce overhead
    access_log off;
    server_tokens off;

    server {
        # PPv2 from OCI LB; higher backlog; reuseport enabled; fastopen OFF
        listen       30080 proxy_protocol reuseport backlog=262144;
        server_name  localhost;

        # Payloads served from /usr/share/nginx/html
        location /payload_ {
            root /usr/share/nginx/html;
        }

        # Health check endpoint
        location /healthz {
            return 200 "ok\\n";
        }

        # Plain-text body with the real client IP from PPv2
        location / {
            return 200 "Client IP: $proxy_protocol_addr\\n";
        }
    }
}"""
    indented_conf = "\n".join("    " + ln for ln in nginx_conf.splitlines())
    cm_yaml = f"""apiVersion: v1
kind: ConfigMap
metadata:
  name: nginx-config
  namespace: {WORK_NS}
data:
  nginx.conf: |
{indented_conf}
"""

    # 3) NGINX DaemonSet (hostNetwork) + Service (LB; Local)
    manifests = f"""apiVersion: apps/v1
kind: DaemonSet
metadata:
  name: nginx-backend
  namespace: {WORK_NS}
  labels:
    app: nginx
spec:
  selector:
    matchLabels:
      app: nginx
  template:
    metadata:
      labels:
        app: nginx
    spec:
      hostNetwork: true
      dnsPolicy: ClusterFirstWithHostNet
      nodeSelector:
        app.role: backend
      initContainers:
      - name: bake-payloads
        image: busybox:1.36
        imagePullPolicy: IfNotPresent
        command: ["/bin/sh","-c"]
        args:
        - |
          set -e
          dd if=/dev/zero of=/payloads/payload_4k   bs=1000 count=4   status=none
          dd if=/dev/zero of=/payloads/payload_10k  bs=1000 count=10  status=none
          dd if=/dev/zero of=/payloads/payload_50k  bs=1000 count=50  status=none
          dd if=/dev/zero of=/payloads/payload_100k bs=1000 count=100 status=none
          dd if=/dev/zero of=/payloads/payload_256k bs=1000 count=256 status=none
          dd if=/dev/zero of=/payloads/payload_1m   bs=1000000 count=1 status=none
          ls -l /payloads
        volumeMounts:
        - name: payloads
          mountPath: /payloads
      containers:
      - name: nginx
        image: index.docker.io/library/nginx:latest
        imagePullPolicy: IfNotPresent
        ports:
        - containerPort: 30080
          hostPort: 30080
        volumeMounts:
        - name: config
          mountPath: /etc/nginx/nginx.conf
          subPath: nginx.conf
        - name: payloads
          mountPath: /usr/share/nginx/html
      volumes:
      - name: config
        configMap:
          name: nginx-config
      - name: payloads
        emptyDir: {{}}
---
apiVersion: v1
kind: Service
metadata:
  name: nginx-service
  namespace: {WORK_NS}
  annotations:
    oci.oraclecloud.com/load-balancer-type: "lb"
    service.beta.kubernetes.io/oci-load-balancer-shape: "flexible"
    service.beta.kubernetes.io/oci-load-balancer-shape-flex-min: "8000"
    service.beta.kubernetes.io/oci-load-balancer-shape-flex-max: "8000"
    service.beta.kubernetes.io/oci-load-balancer-ssl-ports: "443"
    service.beta.kubernetes.io/oci-load-balancer-tls-secret: "ssl-certificate-secret"
    service.beta.kubernetes.io/oci-load-balancer-backend-protocol: "TCP"
    service.beta.kubernetes.io/oci-load-balancer-connection-proxy-protocol-version: "2"
    oci.oraclecloud.com/node-label-selector: "app.role=backend"
spec:
  type: LoadBalancer
  externalTrafficPolicy: Local
  selector:
    app: nginx
  ports:
  - name: https
    port: 443
    targetPort: 30080
    protocol: TCP
"""

    Path("cm_nginx.yaml").write_text(cm_yaml, encoding="utf-8")
    Path("cps_deployment.yaml").write_text(manifests, encoding="utf-8")
    print("Wrote cm_nginx.yaml and cps_deployment.yaml (hostNetwork variant)")

    # 4) Apply NGINX + Service (immutability-safe service recreate), then restart DS to ensure config pickup
    _run(f"kubectl -n {shlex.quote(WORK_NS)} apply -f cm_nginx.yaml", check=True)
    print("\nRecreating Service (immutable-safe) ...")
    _run(f"kubectl -n {shlex.quote(WORK_NS)} delete svc nginx-service --ignore-not-found", check=False)
    _run(f"kubectl -n {shlex.quote(WORK_NS)} apply -f cps_deployment.yaml", check=True)

    print("\nRolling out nginx-backend (restart to load ConfigMap) ...")
    _run(f"kubectl -n {shlex.quote(WORK_NS)} rollout restart ds/nginx-backend", check=True)

    # 5) Wait for NGINX DS rollout
    def rollout_ok(kind, name, timeout=300):
        print(f"\nWaiting for {kind}/{name} rollout ...")
        return _run(f"kubectl -n {shlex.quote(WORK_NS)} rollout status {kind}/{name} --timeout={timeout}s", check=False).returncode == 0

    rollout_ok("ds","nginx-backend", 300)
    _run(f"kubectl -n {shlex.quote(WORK_NS)} get pods -l app=nginx -o wide")
    _run(f"kubectl -n {shlex.quote(WORK_NS)} get endpoints nginx-service -o wide")

    # 6) Wait for EXTERNAL-IP and allow LB convergence
    vip = ""
    deadline = time.time() + 300
    while time.time() < deadline and not vip:
        vip = _jsonpath_svc(WORK_NS, "{.status.loadBalancer.ingress[0].ip}") or _jsonpath_svc(WORK_NS, "{.status.loadBalancer.ingress[0].hostname}")
        if not vip:
            print("Waiting for EXTERNAL-IP ...")
            time.sleep(5)
    print("\nVIP:", vip if vip else "(pending)")
    if not vip:
        raise SystemExit("EXTERNAL-IP still pending; re-run later.")

    print("\nAllowing LB listener/backend convergence (sleep 30s) ...")
    time.sleep(30)

    # 7) Validate /healthz (single + short retry loop)
    print("\nSingle HEAD /healthz (expect 200):")
    _run(f"bash -lc \"curl --http1.1 -skI -w '%{{http_code}}\\n' --connect-timeout 6 --max-time 10 https://{shlex.quote(vip)}/healthz | tail -n1 || true\"")

    print("\nRetry loop (10 HEADs) /healthz (collect http codes):")
    codes = []
    for i in range(10):
        p = _run(f"bash -lc \"curl --http1.1 -skI -w '%{{http_code}}\\n' --connect-timeout 6 --max-time 10 https://{shlex.quote(vip)}/healthz | tail -n1 || true\"", quiet=True)
        codes.append(((p.stdout or '').strip()) or '-')
        time.sleep(0.5)
    print("Codes:", " ".join(codes))

    print(f"\nCell 7 complete (hostNetwork): Tuner applied, NGINX deployed with hostPort 30080, LB ready in ns/{WORK_NS}.")
    print("Proceed to Cell 8 for payload validation and logs; then Cell 9/10 to prepare generators and start CPS tests.")


In [ ]:
# Cell 8 — Objective: Validate VIP endpoints and baked payloads; tail NGINX logs (works with hostNetwork or baseline)
# - Waits for nginx-backend Pod Ready in WORK_NS and prints endpoints
# - Discovers EXTERNAL-IP/hostname (VIP), waits 15s, then:
#   * HEAD /healthz (expect 200)
#   * GET / (expect "Client IP: ...")
#   * GET baked payloads: payload_4k, 10k, 50k, 100k, 256k, 1m (with retries)
# - Tails last 60 lines of NGINX logs
# - Honors RUN_K8S_DEPLOY and WORK_NS

import os, shlex, time, json, subprocess
from pathlib import Path

def _print_cmd(args: list[str]) -> str:
    return " ".join(shlex.quote(a) for a in args)

def _runp(args: list[str], check: bool = False, quiet: bool = False, capture: bool = True):
    if not quiet:
        print(f"\n$ {_print_cmd(args)}")
    p = subprocess.run(args, text=True, capture_output=capture)
    if not quiet:
        if p.stdout:
            print(p.stdout.strip())
        if p.stderr and (("error" in p.stderr.lower()) or ("failed" in p.stderr.lower())):
            print(p.stderr.strip())
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {_print_cmd(args)}\n{p.stdout}\n{p.stderr}")
    return p

def _jsonpath(ns: str, kind: str, name: str, expr: str) -> str:
    args = ["kubectl","-n",ns,"get",kind,name,"-o",f"jsonpath={expr}"]
    p = _runp(args, quiet=True)
    return (p.stdout or "").strip()

def _svc_vip(ns: str) -> str:
    # Try IP, then hostname
    for expr in ("{.status.loadBalancer.ingress[0].ip}", "{.status.loadBalancer.ingress[0].hostname}"):
        args = ["kubectl","-n",ns,"get","svc","nginx-service","-o",f"jsonpath={expr}"]
        p = _runp(args, quiet=True)
        vip = (p.stdout or "").strip()
        if vip:
            return vip
    return ""

RUN_K8S_DEPLOY = os.environ.get("RUN_K8S_DEPLOY", "true").strip().lower() in ("1","true","yes","y")
WORK_NS = os.environ.get("WORK_NS", "lbtest").strip() or "lbtest"

if not RUN_K8S_DEPLOY:
    print("RUN_K8S_DEPLOY is false — skipping VIP/payload validation.")
else:
    # 1) Wait for one nginx-backend pod Ready (up to 180s)
    deadline = time.time() + 180
    ready = False
    pod_name = ""
    while time.time() < deadline and not ready:
        p = _runp(["kubectl","-n",WORK_NS,"get","pods","-l","app=nginx","-o","json"], quiet=True)
        try:
            obj = json.loads(p.stdout or "{}")
            items = obj.get("items", [])
            if items:
                # pick the first; you can extend to all if needed
                pod = items[0]
                pod_name = ((pod.get("metadata") or {}).get("name") or "")
                conds = ((pod.get("status") or {}).get("conditions") or [])
                ready = any(c.get("type")=="Ready" and c.get("status")=="True" for c in conds)
        except Exception:
            pass
        if not ready:
            print("Waiting for nginx-backend pod to be Ready ...")
            time.sleep(4)

    # Show pods and endpoints
    _runp(["kubectl","-n",WORK_NS,"get","pods","-l","app=nginx","-o","wide"])
    _runp(["kubectl","-n",WORK_NS,"get","endpoints","nginx-service","-o","wide"])

    # 2) Discover VIP (IP or hostname)
    vip = _svc_vip(WORK_NS)
    print("\nVIP:", vip if vip else "(pending)")
    if not vip:
        raise SystemExit("EXTERNAL-IP is still pending; wait a bit and re-run this cell.")

    # 3) Quick pause to allow LB convergence after rollout
    print("\nAllowing LB listener/backend convergence (sleep 15s) ...")
    time.sleep(15)

    # 4) Validate /healthz and root via VIP
    print("\n/healthz (headers):")
    _runp(["curl","--http1.1","-skI","--connect-timeout","8","--max-time","12", f"https://{vip}/healthz"], check=False)

    print("\n/ (body should contain 'Client IP: ...'):")
    _runp(["curl","--http1.1","-sk","--connect-timeout","8","--max-time","12", f"https://{vip}/"], check=False)

    # 5) Validate multiple payload sizes via VIP with brief retries
    payloads = [
        ("payload_4k",   4_000),
        ("payload_10k", 10_000),
        ("payload_50k", 50_000),
        ("payload_100k",100_000),
        ("payload_256k",256_000),
        ("payload_1m",1_000_000),
    ]

    print("\nValidating payload sizes (bytes downloaded):")
    summary = []
    for name, expected_min in payloads:
        tmpfile = f"/tmp/{name}.out"
        ok = False
        for attempt in range(1, 5):  # up to 4 attempts
            # Clean and fetch
            _runp(["bash","-lc", f"rm -f {shlex.quote(tmpfile)}"], quiet=True)
            _runp(["curl","--http1.1","-sk","--connect-timeout","10","--max-time","20",
                   "-o", tmpfile, f"https://{vip}/{name}"], check=False, quiet=True)
            # Count bytes
            p_wc = _runp(["bash","-lc", f"test -f {shlex.quote(tmpfile)} && wc -c < {shlex.quote(tmpfile)} || echo 0"], quiet=True)
            try:
                bytes_dl = int((p_wc.stdout or "0").strip())
            except Exception:
                bytes_dl = 0
            print(f"  {name}: attempt {attempt} -> {bytes_dl} bytes")
            if bytes_dl > 0:
                ok = True
                summary.append((name, bytes_dl))
                break
            time.sleep(3)
        if not ok:
            summary.append((name, 0))

    # 6) NGINX logs tail to confirm requests hit the pod
    print("\n== NGINX logs (last 60 lines) ==")
    if not pod_name:
        # try resolve again
        pod_name = _jsonpath(WORK_NS, "pods", "-l app=nginx", "{.items[0].metadata.name}") or ""
    if pod_name:
        _runp(["kubectl","-n",WORK_NS,"logs",pod_name,"--tail","60"], check=False)
    else:
        print("No nginx pod name resolved for logs tail.")

    # 7) Print summary
    print("\nSummary:")
    print(f"  VIP={vip}")
    for name, bytes_dl in summary:
        print(f"  {name}: {bytes_dl} bytes")

    print("\nCell 8 complete: VIP and payloads validated; logs tailed. Proceed to Cell 9 to prepare generators, then Cell 10 to start CPS tests.")


In [ ]:
# Cell 9 — Objective: Prepare generators with an isolated virtual environment (venv) and pinned versions + ensure tmux is installed
# - Creates/refreshes ~/locustvenv on each generator host (IPv4 SSH pinned)
# - Installs pinned packages in venv: locust==2.43.3, urllib3==2.6.3 (adjust pins as desired)
# - Installs tmux via dnf/yum/microdnf if missing
# - Verifies: python version, venv site-packages, locust.__version__, urllib3.__version__,
#             and urllib3.util.ssl_.create_urllib3_context, plus tmux -V
# - Optional: set RESET_VENV=true in environment before running this cell to force a clean venv

import os, json, paramiko, concurrent.futures
from pathlib import Path

def _load_list_env(name: str, default="[]"):
    try:
        v = json.loads(os.environ.get(name, default))
        return v if isinstance(v, list) else []
    except Exception:
        return []

GEN_IPS = _load_list_env("GEN_IPS_V4_JSON")
if not GEN_IPS:
    raise RuntimeError("GEN_IPS_V4_JSON is empty. Ensure Cell 5 exported generator IPv4s.")

SSH_KEY  = os.path.expanduser(os.environ.get("SSH_PRIVATE_KEY_PATH", "").strip())
if not SSH_KEY or not Path(SSH_KEY).exists():
    raise FileNotFoundError(f"SSH_PRIVATE_KEY_PATH not set or not found. Current: {SSH_KEY or '(unset)'}")
SSH_PASS = os.environ.get("SSH_KEY_PASSPHRASE", None)

RESET_VENV = os.environ.get("RESET_VENV", "false").strip().lower() in ("1","true","yes","y")

def _load_pkey(path, pw):
    for Key in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
        try:
            return Key.from_private_key_file(path, password=pw)
        except Exception:
            pass
    raise paramiko.SSHException(f"Could not load SSH private key. Path: {path}")

def ssh_exec(host: str, command: str, timeout: int = 600):
    pkey = _load_pkey(SSH_KEY, SSH_PASS)
    cli = paramiko.SSHClient()
    cli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cli.connect(hostname=host, username="opc", pkey=pkey, timeout=30)
    full = f"bash -lc {json.dumps(command)}"
    _, stdout, stderr = cli.exec_command(full, timeout=timeout)
    out = (stdout.read().decode("utf-8","ignore") or "").strip()
    err = (stderr.read().decode("utf-8","ignore") or "").strip()
    try:
        rc = stdout.channel.recv_exit_status()
    except Exception:
        rc = None
    cli.close()
    return rc, out, err

PREP_CONCURRENCY = max(1, int(os.environ.get("PREP_CONCURRENCY","5")))

# 0) Stop any running locust processes (best-effort)
CMD_STOP = r"pkill -f 'python3 -m locust' 2>/dev/null || true; pkill -f 'locust .*--worker' 2>/dev/null || true; pkill -f 'locust .*--master' 2>/dev/null || true"

# 1) Optionally reset venv
CMD_VENV_RESET = 'rm -rf "$HOME/locustvenv"'

# 2) Create venv (fallback to ensurepip if venv module is missing), then upgrade base tooling and install pinned packages
CMD_VENV_CREATE = (
    'python3 -m venv "$HOME/locustvenv" || (python3 -m ensurepip --upgrade && python3 -m venv "$HOME/locustvenv")'
)
CMD_VENV_UPGRADE = (
    'source "$HOME/locustvenv/bin/activate" && '
    'python -m pip install --upgrade --no-cache-dir pip setuptools wheel && '
    'python -m pip install --upgrade --no-cache-dir "locust==2.43.3" "urllib3==2.6.3"'
)

# 3) Verify inside venv using python -c (robust quoting)
CMD_VENV_VERIFY = (
    "source \"$HOME/locustvenv/bin/activate\" && "
    "python -c "
    "'import sys,site,locust,urllib3; "
    "print(\"python\", sys.version.split()[0]); "
    "print(\"venv_site\", site.getsitepackages()); "
    "print(\"locust\", locust.__version__); "
    "print(\"urllib3\", urllib3.__version__); "
    "import urllib3.util.ssl_ as sslmod; "
    "print(\"sslmod_has_symbol:\", hasattr(sslmod, \"create_urllib3_context\"))'"
)

# 4) Ensure tmux present, then verify
CMD_TMUX_INSTALL = (
    'command -v tmux >/dev/null 2>&1 || '
    '( (command -v dnf >/dev/null 2>&1 && sudo -n dnf -y install tmux) || '
    '  (command -v yum >/dev/null 2>&1 && sudo -n yum -y install tmux) || '
    '  (command -v microdnf >/dev/null 2>&1 && sudo -n microdnf -y install tmux) || '
    '  (echo "tmux install failed (no known pkg mgr)" >&2) )'
)
CMD_TMUX_VERIFY = 'tmux -V || echo "tmux not installed"'

print(f"Preparing venv and tmux on {len(GEN_IPS)} generator(s) in parallel (RESET_VENV={RESET_VENV}) ...")
results = {}
with concurrent.futures.ThreadPoolExecutor(max_workers=min(PREP_CONCURRENCY, len(GEN_IPS))) as ex:
    futs = []
    for h in GEN_IPS:
        def _do(host: str):
            logs = []
            # stop
            logs.append(("stop",) + ssh_exec(host, CMD_STOP))
            # reset (optional)
            if RESET_VENV:
                logs.append(("venv_reset",) + ssh_exec(host, CMD_VENV_RESET))
            # create venv
            logs.append(("venv_create",) + ssh_exec(host, CMD_VENV_CREATE))
            # upgrade/install pins
            logs.append(("venv_upgrade",) + ssh_exec(host, CMD_VENV_UPGRADE))
            # verify venv
            logs.append(("venv_verify",) + ssh_exec(host, CMD_VENV_VERIFY))
            # install tmux if missing
            logs.append(("tmux_install",) + ssh_exec(host, CMD_TMUX_INSTALL))
            # verify tmux
            logs.append(("tmux_verify",) + ssh_exec(host, CMD_TMUX_VERIFY, timeout=60))
            return host, logs
        futs.append(ex.submit(_do, h))
    for f in futs:
        host, logs = f.result()
        results[host] = logs

ok_hosts, bad_hosts = [], []
for host, logs in results.items():
    print(f"\n==== {host} ====")
    venv_verify = next(((rc, out, err) for (tag, rc, out, err) in logs if tag == "venv_verify"), (None,"",""))
    tmux_verify = next(((rc, out, err) for (tag, rc, out, err) in logs if tag == "tmux_verify"), (None,"",""))
    print(venv_verify[1] or venv_verify[2] or f"(venv_verify_rc={venv_verify[0]})")
    print(tmux_verify[1] or tmux_verify[2] or f"(tmux_verify_rc={tmux_verify[0]})")
    if "sslmod_has_symbol: True" in (venv_verify[1] or "") and "tmux" in (tmux_verify[1] or tmux_verify[2] or ""):
        ok_hosts.append(host)
    else:
        bad_hosts.append(host)

print("\nVenv+tmux prepare summary:")
print(f"  OK hosts: {len(ok_hosts)}/{len(GEN_IPS)}")
if bad_hosts:
    print("  Hosts needing attention:", bad_hosts)
else:
    print("  All hosts have venv pins and tmux present.")

print("\nCell 9 complete: Generators ready with venv + tmux. NEXT: Run Cell 10 to start Locust master (tmux) and workers.")


In [ ]:
# Cell 10 — Objective: Start Locust (master UI + workers) using per-host venv python
# - Uses $HOME/locustvenv/bin/python on each generator (isolated, pinned deps from Cell 9)
# - Auto-targets VIP if EXTERNAL_TARGETS_TEXT empty (reads nginx-service in ns/WORK_NS)
# - Starts master (tmux ui_master) and spawns per-host workers; waits for readiness
# - Prints direct URL and SSH tunnel hint for the UI

import os, json, time, shlex, paramiko, concurrent.futures, subprocess
from pathlib import Path
from typing import List, Tuple

def _load_list_env(name: str, default="[]") -> List[str]:
    try:
        v = json.loads(os.environ.get(name, default))
        return v if isinstance(v, list) else []
    except Exception:
        return []

WORK_NS = os.environ.get("WORK_NS", "lbtest").strip() or "lbtest"

def _svc_vip(ns: str) -> str:
    for expr in ("{.status.loadBalancer.ingress[0].ip}", "{.status.loadBalancer.ingress[0].hostname}"):
        cmd = ["kubectl","-n",ns,"get","svc","nginx-service","-o",f"jsonpath={expr}"]
        try:
            p = subprocess.run(cmd, text=True, capture_output=True)
            vip = (p.stdout or "").strip()
            if vip:
                return vip
        except Exception:
            pass
    return ""

GEN_IPS_V4 = _load_list_env("GEN_IPS_V4_JSON")
GEN_IPS_V6 = _load_list_env("GEN_IPS_V6_JSON")
USE_SSH_IPV6 = os.environ.get("USE_SSH_IPV6", "false").strip().lower() in ("1","true","yes","y")
GEN_IPS = GEN_IPS_V6 if (USE_SSH_IPV6 and GEN_IPS_V6) else GEN_IPS_V4
if not GEN_IPS:
    raise RuntimeError("No generator hosts available. Ensure Cell 5 completed and GEN_IPS env vars are present.")

MASTER = GEN_IPS[0]
VENV_PY = "$HOME/locustvenv/bin/python"

# Runtime settings (from Cell 3)
TEST_MODE                = os.environ.get("TEST_MODE", "cps").strip().lower()
LOCUST_WAIT_TIME_S       = os.environ.get("LOCUST_WAIT_TIME_SEC", "1.0")
LOCUST_CONNECT_MS        = os.environ.get("LOCUST_CONNECT_TIMEOUT_MS", "8000")
LOCUST_READ_MS           = os.environ.get("LOCUST_READ_TIMEOUT_MS",  "15000")
LOCUST_VERIFY_TLS        = (os.environ.get("LOCUST_VERIFY_TLS", "false").strip().lower() == "true")
HEALTH_ENDPOINT_PATH     = os.environ.get("HEALTH_ENDPOINT_PATH", "/healthz")
THROUGHPUT_ENDPOINT_PATH = os.environ.get("THROUGHPUT_ENDPOINT_PATH", "/payload_100k")
UI_WEB_PORT              = int(os.environ.get("UI_WEB_PORT", "8089"))
EXTERNAL_TARGETS_TEXT    = os.environ.get("EXTERNAL_TARGETS_TEXT", "").strip()
LOCUST_WORKDIR           = os.path.expanduser(os.environ.get("LOCUST_WORKDIR", "/home/opc/locustwork"))

# Worker policy
WORKERS_PER_HOST_ENV   = os.environ.get("WORKERS_PER_HOST", "auto").strip().lower()
CPU_RESERVE            = int(os.environ.get("CPU_RESERVE", "1"))
MIN_WORKERS_PER_HOST   = int(os.environ.get("MIN_WORKERS_PER_HOST", "1"))
MAX_WORKERS_PER_HOST   = int(os.environ.get("MAX_WORKERS_PER_HOST", "32"))

def _ms_to_s_str(ms_str: str, default_s: str) -> str:
    try:
        return str(round(max(0.001, float(ms_str) / 1000.0), 3))
    except Exception:
        return default_s

CONNECT_S = _ms_to_s_str(LOCUST_CONNECT_MS, "8")
READ_S    = _ms_to_s_str(LOCUST_READ_MS, "15")

SSH_KEY  = os.path.expanduser(os.environ.get("SSH_PRIVATE_KEY_PATH", "").strip() or "~/.ssh/id_rsa")
if not Path(SSH_KEY).exists():
    raise FileNotFoundError(f"SSH_PRIVATE_KEY_PATH not found: {SSH_KEY}")
SSH_PASS = os.environ.get("SSH_KEY_PASSPHRASE", None)

def _load_pkey(path: str, pw: str | None):
    for Key in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
        try:
            return Key.from_private_key_file(path, password=pw)
        except Exception:
            pass
    raise paramiko.SSHException(f"Could not load SSH private key: {path}")

def ssh_exec(host: str, command: str, timeout: int = 90) -> Tuple[str, str, int]:
    pkey = _load_pkey(SSH_KEY, SSH_PASS)
    cli = paramiko.SSHClient()
    cli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cli.connect(hostname=host, username="opc", pkey=pkey, timeout=30, banner_timeout=30, auth_timeout=30)
    full = f"bash -lc {json.dumps(command)}"
    _, stdout, stderr = cli.exec_command(full, timeout=timeout)
    out = (stdout.read().decode("utf-8", "ignore") or "").strip()
    err = (stderr.read().decode("utf-8", "ignore") or "").strip()
    try:
        rc = stdout.channel.recv_exit_status()
    except Exception:
        rc = None
    cli.close()
    return out, err, rc

def sftp_write(host: str, content: str, remote_path: str, mode: int = 0o755):
    pkey = _load_pkey(SSH_KEY, SSH_PASS)
    cli = paramiko.SSHClient()
    cli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cli.connect(hostname=host, username="opc", pkey=pkey, timeout=30, banner_timeout=30, auth_timeout=30)
    sftp = cli.open_sftp()
    try:
        parent = os.path.dirname(remote_path)
        try:
            sftp.stat(parent)
        except FileNotFoundError:
            sftp.mkdir(parent)
        with sftp.file(remote_path, "w") as f:
            f.write(content)
        sftp.chmod(remote_path, mode)
    finally:
        sftp.close(); cli.close()

# Auto-target VIP if user didn't provide explicit targets
TARGETS_CSV = EXTERNAL_TARGETS_TEXT
if not TARGETS_CSV:
    vip = _svc_vip(WORK_NS)
    if vip:
        TARGETS_CSV = f"https://{vip}"

def _detect_master_bind_and_addr() -> Tuple[str, str]:
    out, _, _ = ssh_exec(MASTER, r"ip -4 route get 1.1.1.1 2>/dev/null | awk '{for(i=1;i<=NF;i++) if($i==\"src\") {print $(i+1); exit}}'", timeout=10)
    ip = (out or "").strip()
    if not ip:
        out, _, _ = ssh_exec(MASTER, r"hostname -I 2>/dev/null | tr ' ' '\n' | awk '/^[0-9]+\.[0-9]+\.[0-9]+\.[0-9]+$/{print; exit}'", timeout=10)
        ip = (out or "").strip()
    if not ip:
        out, _, _ = ssh_exec(MASTER, r"ip -4 addr show scope global 2>/dev/null | awk '/inet /{print $2}' | cut -d/ -f1 | head -n1", timeout=10)
        ip = (out or "").strip()
    return ("0.0.0.0", (ip or MASTER))

MASTER_BIND_HOST, MASTER_ADDR_FOR_WORKERS_RAW = _detect_master_bind_and_addr()
if " " in (MASTER_ADDR_FOR_WORKERS_RAW or ""):
    toks = (MASTER_ADDR_FOR_WORKERS_RAW or "").split()
    MASTER_ADDR_FOR_WORKERS_RAW = next((t for t in toks if t.count(".")==3), MASTER)
MASTER_ADDR_FOR_WORKERS = shlex.quote(MASTER_ADDR_FOR_WORKERS_RAW)

def _env_exports() -> str:
    return "\n".join([
        f"export LOCUST_MODE={shlex.quote(TEST_MODE)}",
        f"export LOCUST_HEALTH_PATH={shlex.quote(HEALTH_ENDPOINT_PATH)}",
        f"export LOCUST_THROUGHPUT_PATH={shlex.quote(THROUGHPUT_ENDPOINT_PATH)}",
        f"export LOCUST_VERIFY_TLS={'true' if LOCUST_VERIFY_TLS else 'false'}",
        f"export LOCUST_CONNECT_TIMEOUT_S={CONNECT_S}",
        f"export LOCUST_READ_TIMEOUT_S={READ_S}",
        f"export LOCUST_WAIT_TIME_S={LOCUST_WAIT_TIME_S}",
        'export PATH="$HOME/.local/bin:/usr/local/bin:/usr/bin:/bin:$PATH"',
        (f"export LOCUST_TARGETS={shlex.quote(TARGETS_CSV)}\nexport LOCUST_NAME_BY_VIP=true" if TARGETS_CSV else "unset LOCUST_TARGETS\nexport LOCUST_NAME_BY_VIP=false"),
    ])

# Master: tmux session (ui_master), foreground exec using venv python
run_ui_sh = f"""#!/usr/bin/env bash
set -euo pipefail
{_env_exports()}
cd {shlex.quote(LOCUST_WORKDIR)}
exec {VENV_PY} -m locust -f locustfile.py --master --master-bind-host {MASTER_BIND_HOST} \
  --web-host 0.0.0.0 --web-port {UI_WEB_PORT} \
  > {shlex.quote(LOCUST_WORKDIR)}/ui_master.log 2>&1
"""
sftp_write(MASTER, run_ui_sh, f"{LOCUST_WORKDIR}/run_ui.sh", mode=0o755)
ssh_exec(MASTER, "tmux has-session -t ui_master 2>/dev/null && tmux kill-session -t ui_master || true", timeout=10)
_, err, rc = ssh_exec(MASTER, f"tmux new -d -s ui_master {shlex.quote(LOCUST_WORKDIR)}/run_ui.sh", timeout=20)
print(f"[master@{MASTER}] ui_master started (tmux rc={rc})")
if err: print("[stderr]", err)

# Compute workers per host
def _detect_nproc(host: str) -> int:
    out, _, _ = ssh_exec(host, "nproc 2>/dev/null || getconf _NPROCESSORS_ONLN 2>/dev/null || echo 1", timeout=10)
    try:
        return max(1, int((out or "1").strip()))
    except Exception:
        return 1

def _workers_for_host(host: str) -> int:
    if WORKERS_PER_HOST_ENV != "auto":
        try:
            return max(1, int(WORKERS_PER_HOST_ENV))
        except Exception:
            return 1
    n = _detect_nproc(host)
    w = max(1, n - max(0, CPU_RESERVE))
    return max(MIN_WORKERS_PER_HOST, min(MAX_WORKERS_PER_HOST, w))

per_host_workers = {h: _workers_for_host(h) for h in GEN_IPS}
print(f"Planned workers per host: {per_host_workers} | total={sum(per_host_workers.values())}")

# Per-host start script using nohup; venv python
def spawn_on_host(host: str) -> Tuple[str, int, str]:
    n = per_host_workers[host]
    start_workers_sh = f"""#!/usr/bin/env bash
set -euo pipefail
{_env_exports()}
cd {shlex.quote(LOCUST_WORKDIR)}
echo "Spawning {n} worker(s) to master {MASTER_ADDR_FOR_WORKERS_RAW} at $(date -u)"
for i in $(seq 1 {n}); do
  nohup {VENV_PY} -m locust -f locustfile.py --worker --master-host {MASTER_ADDR_FOR_WORKERS} > locust-worker-$i.log 2>&1 &
  sleep 0.05
done
echo WORKERS_STARTED $(date -u)
"""
    remote_path = f"{LOCUST_WORKDIR}/start_workers.sh"
    sftp_write(host, start_workers_sh, remote_path, mode=0o755)
    out, err, rc = ssh_exec(host, f"nohup {shlex.quote(remote_path)} > {shlex.quote(LOCUST_WORKDIR)}/start_workers.out 2>&1 & echo OK", timeout=15)
    return host, (0 if "OK" in out else (rc if rc is not None else 1)), err

print(f"Spawning workers (parallel hosts; concurrency={min(8, len(GEN_IPS))}) ...")
with concurrent.futures.ThreadPoolExecutor(max_workers=min(8, len(GEN_IPS))) as ex:
    for host, rc, err in ex.map(spawn_on_host, GEN_IPS):
        if rc != 0 or err:
            print(f"[host-spawn][{host}] rc={rc} err={err}")

# Readiness: wait for master UI and per-host workers
def wait_master_ui(timeout_s=90, interval_s=1.5):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        out, _, _ = ssh_exec(MASTER, f"ss -lntp | grep -E ':{UI_WEB_PORT}\\s' || true", timeout=10)
        if (out or "").strip():
            return True, out.strip()
        time.sleep(interval_s)
    return False, ""

ok, listen_info = wait_master_ui()
print("\nMaster UI readiness:", "READY" if ok else "NOT READY")
if listen_info:
    print(listen_info)
else:
    log_tail, _, _ = ssh_exec(MASTER, f'tail -n 60 "{LOCUST_WORKDIR}/ui_master.log" 2>/dev/null || true', timeout=10)
    if log_tail:
        print("Master log (tail):")
        print(log_tail)

def wait_workers(host: str, expected: int, timeout_s=180, interval_s=2):
    deadline = time.time() + timeout_s
    last = "0"
    while time.time() < deadline:
        cnt, _, _ = ssh_exec(host, r"pgrep -f 'locust.*--worker' | wc -l || true", timeout=10)
        last = (cnt or "0").strip()
        try:
            if int(last) >= expected:
                return True, last
        except Exception:
            pass
        time.sleep(interval_s)
    return False, last

print("\nWorker readiness per host:")
for hip in GEN_IPS:
    exp = per_host_workers[hip]
    ready, seen = wait_workers(hip, exp, timeout_s=180, interval_s=2)
    print(f"  {hip}: {'READY' if ready else 'NOT READY'} (seen={seen}, expected~{exp})")
    if not ready:
        wlog, _, _ = ssh_exec(hip, 'ls -1t "$HOME/locustwork"/locust-worker-*.log 2>/dev/null | head -n1 | xargs -r tail -n 30 || true', timeout=10)
        if wlog:
            print("    worker log (tail):")
            print("    " + "\n    ".join(wlog.splitlines()))

# UI URL and SSH tunnel
print("\nLocust UI:")
print(f"  Direct URL (public):   http://{MASTER}:{UI_WEB_PORT}")
print("  Recommended (tunnel) from your laptop:")
print(f'    ssh -i "{SSH_KEY}" -o ExitOnForwardFailure=yes -N -L {UI_WEB_PORT}:127.0.0.1:{UI_WEB_PORT} opc@{MASTER}')
print("  Then open:")
print(f"    http://127.0.0.1:{UI_WEB_PORT}")

print("\nCell 10 complete: Master up; workers spawned using per-host venv. Use the UI to start CPS runs.")


In [ ]:
# Cell 11 — Objective: Export Locust statistics (CSV) from master UI and capture quick diagnostics
# - Pulls /stats/requests/csv, /stats/distribution/csv, /exceptions/csv from the master UI
# - Saves under ./results using TS_UTC and a short run tag
# - Captures a compact summary.json and the last 80 lines of the master log

import os, json, time, shlex, paramiko
from pathlib import Path
from typing import Tuple

# Discover master and UI port from prior cells
GEN_IPS_V4 = json.loads(os.environ.get("GEN_IPS_V4_JSON","[]") or "[]")
if not GEN_IPS_V4:
    raise RuntimeError("GEN_IPS_V4_JSON not present. Ensure Cell 5 and Cell 10 ran.")
MASTER = GEN_IPS_V4[0]
UI_WEB_PORT = int(os.environ.get("UI_WEB_PORT", "8089"))
LOCUST_WORKDIR = os.path.expanduser(os.environ.get("LOCUST_WORKDIR", "/home/opc/locustwork"))

# Output folder and timestamp
OUTPUT_DIR = Path(os.path.abspath(os.environ.get("OUTPUT_DIR", "./results")))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TS_UTC = os.environ.get("TS_UTC") or time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
os.environ["TS_UTC"] = TS_UTC

# SSH auth
SSH_KEY  = os.path.expanduser(os.environ.get("SSH_PRIVATE_KEY_PATH", "").strip() or "~/.ssh/id_rsa")
if not Path(SSH_KEY).exists():
    raise FileNotFoundError(f"SSH_PRIVATE_KEY_PATH not found: {SSH_KEY}")
SSH_PASS = os.environ.get("SSH_KEY_PASSPHRASE", None)

def _load_pkey(path: str, pw: str | None):
    for Key in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
        try:
            return Key.from_private_key_file(path, password=pw)
        except Exception:
            pass
    raise paramiko.SSHException(f"Could not load SSH private key: {path}")

def ssh_exec(host: str, command: str, timeout: int = 90) -> Tuple[str, str, int]:
    pkey = _load_pkey(SSH_KEY, SSH_PASS)
    cli = paramiko.SSHClient()
    cli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cli.connect(hostname=host, username="opc", pkey=pkey, timeout=30, banner_timeout=30, auth_timeout=30)
    full = f"bash -lc {json.dumps(command)}"
    _, stdout, stderr = cli.exec_command(full, timeout=timeout)
    out = (stdout.read().decode("utf-8", "ignore") or "").strip()
    err = (stderr.read().decode("utf-8", "ignore") or "").strip()
    try:
        rc = stdout.channel.recv_exit_status()
    except Exception:
        rc = None
    cli.close()
    return out, err, rc

# Pull CSVs to the master host /tmp, then SFTP them back
def pull_csv(filename: str, endpoint: str) -> Path:
    remote_tmp = f"/tmp/{filename}"
    curl_cmd = f"curl -sS http://127.0.0.1:{UI_WEB_PORT}{endpoint} -o {shlex.quote(remote_tmp)} && test -s {shlex.quote(remote_tmp)} && echo OK || echo FAIL"
    out, err, rc = ssh_exec(MASTER, curl_cmd, timeout=30)
    if "FAIL" in out or rc not in (0, None):
        raise RuntimeError(f"CSV download failed for {endpoint} on master: rc={rc}, out={out}, err={err}")
    # SFTP back
    pkey = _load_pkey(SSH_KEY, SSH_PASS)
    cli = paramiko.SSHClient()
    cli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cli.connect(hostname=MASTER, username="opc", pkey=pkey, timeout=30, banner_timeout=30, auth_timeout=30)
    sftp = cli.open_sftp()
    try:
        local_path = OUTPUT_DIR / filename
        sftp.get(remote_tmp, str(local_path))
    finally:
        sftp.close(); cli.close()
    return local_path

# Filenames with timestamp
requests_csv = f"locust_requests_{TS_UTC}.csv"
distribution_csv = f"locust_distribution_{TS_UTC}.csv"
exceptions_csv = f"locust_exceptions_{TS_UTC}.csv"

print("Pulling Locust CSVs from master ...")
p_req = pull_csv(requests_csv, "/stats/requests/csv")
p_dist = pull_csv(distribution_csv, "/stats/distribution/csv")
p_ex = pull_csv(exceptions_csv, "/exceptions/csv")

# Master log tail
print("\nMaster log (last 80 lines):")
log_tail, _, _ = ssh_exec(MASTER, f'tail -n 80 "{LOCUST_WORKDIR}/ui_master.log" 2>/dev/null || true', timeout=15)
print(log_tail)

# Compact summary JSON
summary = {
    "master": MASTER,
    "ui_port": UI_WEB_PORT,
    "ts_utc": TS_UTC,
    "files": {
        "requests_csv": str(p_req),
        "distribution_csv": str(p_dist),
        "exceptions_csv": str(p_ex),
    }
}
sum_path = OUTPUT_DIR / f"locust_summary_{TS_UTC}.json"
sum_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("\nSaved CSVs and summary to:")
print(" ", p_req)
print(" ", p_dist)
print(" ", p_ex)
print(" ", sum_path)

print("\nCell 11 complete: Results exported. You can re-run this cell anytime during/after the test to snapshot CSVs.")


In [ ]:
# Cell 12 — Objective: Stop Locust UI and workers cleanly across all generators
# - Kills the master tmux session (ui_master) on the master host
# - Kills all worker processes on every generator host in parallel
# - Verifies remaining worker counts; confirms master UI is not listening
# - Writes a concise stop summary JSON to ./results using TS_UTC

import os, json, time, paramiko, concurrent.futures
from pathlib import Path
from typing import List, Tuple

def _load_list_env(name: str, default="[]") -> List[str]:
    try:
        v = json.loads(os.environ.get(name, default))
        return v if isinstance(v, list) else []
    except Exception:
        return []

# Resolve generator hosts and master
GEN_IPS_V4 = _load_list_env("GEN_IPS_V4_JSON")
GEN_IPS_V6 = _load_list_env("GEN_IPS_V6_JSON")
USE_SSH_IPV6 = os.environ.get("USE_SSH_IPV6", "false").strip().lower() in ("1","true","yes","y")
GEN_IPS = GEN_IPS_V6 if (USE_SSH_IPV6 and GEN_IPS_V6) else GEN_IPS_V4
if not GEN_IPS:
    raise RuntimeError("No generator hosts available. Ensure Cell 5 and Cell 10 ran.")
MASTER = GEN_IPS[0]

UI_WEB_PORT = int(os.environ.get("UI_WEB_PORT", "8089"))
LOCUST_WORKDIR = os.path.expanduser(os.environ.get("LOCUST_WORKDIR", "/home/opc/locustwork"))

# Output directory and timestamp
OUTPUT_DIR = Path(os.path.abspath(os.environ.get("OUTPUT_DIR", "./results")))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TS_UTC = os.environ.get("TS_UTC") or time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
os.environ["TS_UTC"] = TS_UTC

# SSH authentication
SSH_KEY  = os.path.expanduser(os.environ.get("SSH_PRIVATE_KEY_PATH", "").strip() or "~/.ssh/id_rsa")
if not Path(SSH_KEY).exists():
    raise FileNotFoundError(f"SSH_PRIVATE_KEY_PATH not found: {SSH_KEY}")
SSH_PASS = os.environ.get("SSH_KEY_PASSPHRASE", None)

def _load_pkey(path: str, pw: str | None):
    for Key in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
        try:
            return Key.from_private_key_file(path, password=pw)
        except Exception:
            pass
    raise paramiko.SSHException(f"Could not load SSH private key: {path}")

def ssh_exec(host: str, command: str, timeout: int = 90) -> Tuple[str, str, int]:
    pkey = _load_pkey(SSH_KEY, SSH_PASS)
    cli = paramiko.SSHClient()
    cli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cli.connect(hostname=host, username="opc", pkey=pkey, timeout=30, banner_timeout=30, auth_timeout=30)
    full = f"bash -lc {json.dumps(command)}"
    _, stdout, stderr = cli.exec_command(full, timeout=timeout)
    out = (stdout.read().decode("utf-8", "ignore") or "").strip()
    err = (stderr.read().decode("utf-8", "ignore") or "").strip()
    try:
        rc = stdout.channel.recv_exit_status()
    except Exception:
        rc = None
    cli.close()
    return out, err, rc

# 1) Stop master UI (tmux) on the master host and kill any master process (best-effort)
print(f"[master@{MASTER}] Stopping master UI (tmux ui_master) ...")
ssh_exec(MASTER, "tmux has-session -t ui_master 2>/dev/null && tmux kill-session -t ui_master || true", timeout=15)
# Also best-effort kill master process if it wasn't in tmux
ssh_exec(MASTER, 'pkill -f "locust .*--master" 2>/dev/null || true; pkill -f "python3 -m locust .*--master" 2>/dev/null || true', timeout=15)

# 2) Stop workers across all hosts in parallel
def stop_workers_on_host(host: str) -> Tuple[str, int]:
    # Kill both venv and non-venv worker invocations (best-effort)
    cmd = (
        'pkill -f "python3 -m locust .*--worker" 2>/dev/null || true; '
        'pkill -f "locust .*--worker" 2>/dev/null || true; '
        'pgrep -f "locust.*--worker" | wc -l || true'
    )
    out, err, rc = ssh_exec(host, cmd, timeout=20)
    try:
        remaining = int((out or "0").splitlines()[-1].strip())
    except Exception:
        remaining = 0
    return host, remaining

print("Stopping workers on all generator hosts ...")
remaining_by_host = {}
with concurrent.futures.ThreadPoolExecutor(max_workers=min(8, len(GEN_IPS))) as ex:
    for host, remaining in ex.map(stop_workers_on_host, GEN_IPS):
        remaining_by_host[host] = remaining
        print(f"  {host}: remaining workers ≈ {remaining}")

# 3) Confirm master UI is not listening anymore (should be empty)
print("\nVerifying master UI listener is closed ...")
out, _, _ = ssh_exec(MASTER, f"ss -lntp | grep -E ':{UI_WEB_PORT}\\s' || true", timeout=10)
if (out or "").strip():
    print("WARNING: UI still appears to be listening:")
    print(out.strip())
else:
    print("Master UI listener closed.")

# 4) Save a concise stop summary
summary = {
    "ts_utc": TS_UTC,
    "master": MASTER,
    "ui_port": UI_WEB_PORT,
    "remaining_workers": remaining_by_host,
    "ui_listen_present": bool((out or "").strip()),
}
sum_path = OUTPUT_DIR / f"locust_stop_summary_{TS_UTC}.json"
sum_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("\nStop summary written to:", sum_path)

print("\nCell 12 complete: Locust UI and workers stopped.")
print("You can re-run Cell 10 to start a new test, or proceed to Cell 13 to teardown infrastructure.")


In [ ]:
# Cell 13 — Objective: Simple one-shot teardown for this notebook's stack
# - Best-effort K8s cleanup (Service/DS/CM + namespace)
# - Terraform clean/init and single full destroy using stack/terraform.tfvars

import os, subprocess, shlex, time
from pathlib import Path

def _run(cmd: str, cwd: Path | None = None, check: bool = True, echo: bool = True) -> int:
    if echo:
        print(f"\n$ {cmd}")
    p = subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None, text=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}")
    return p.returncode

def _which(name: str) -> bool:
    try:
        import shutil
        return shutil.which(name) is not None
    except Exception:
        return False

# Context
WORK_NS   = os.environ.get("WORK_NS", "lbtest").strip() or "lbtest"
stack_dir = Path("stack").resolve()
tfvars    = stack_dir / "terraform.tfvars"

if not stack_dir.is_dir():
    raise RuntimeError(f"Terraform working directory not found: {stack_dir}")
if not tfvars.exists():
    raise RuntimeError(f"terraform.tfvars not found in {stack_dir}; ensure Cells 4/5 ran successfully.")

print("=== Simple Teardown Start ===")
print(f"Namespace (best-effort cleanup): {WORK_NS}")
print(f"Terraform dir: {stack_dir}")

# 1) K8s cleanup (best-effort; skip if kubectl is not present)
if _which("kubectl"):
    try:
        # Delete LB Service first; this helps release the OCI LB before VCN teardown
        _run(f"kubectl -n {shlex.quote(WORK_NS)} delete svc nginx-service --ignore-not-found", check=False)
        # Delete DS + ConfigMap (ignore if missing)
        _run(f"kubectl -n {shlex.quote(WORK_NS)} delete ds nginx-backend --ignore-not-found", check=False)
        _run(f"kubectl -n {shlex.quote(WORK_NS)} delete ds ds-backend-tuner --ignore-not-found", check=False)
        _run(f"kubectl -n {shlex.quote(WORK_NS)} delete cm nginx-config --ignore-not-found", check=False)
        # Small wait for LB release
        print("Waiting briefly for Service/LB cleanup (10s) ...")
        time.sleep(10)
        # Delete namespace (no hard wait; simple and fast)
        _run(f"kubectl delete namespace {shlex.quote(WORK_NS)} --ignore-not-found --timeout=60s --wait=false", check=False)
    except Exception as e:
        print(f"[note] K8s cleanup encountered an error (continuing): {e}")
else:
    print("[note] kubectl not found; skipping K8s cleanup")

# 2) Terraform: clean artifacts and destroy
# Remove any stale lock/plugins to keep it simple and deterministic
lockfile = stack_dir / ".terraform.lock.hcl"
tf_dir   = stack_dir / ".terraform"
try:
    if lockfile.exists():
        print(f"Removing {lockfile}")
        lockfile.unlink()
    if tf_dir.exists() and tf_dir.is_dir():
        print(f"Removing {tf_dir}")
        _run(f"rm -rf {shlex.quote(str(tf_dir))}", cwd=stack_dir, check=False)
except Exception as e:
    print(f"[note] Could not clean terraform artifacts: {e}")

# Init and one-shot destroy
_run("terraform init -upgrade -input=false", cwd=stack_dir, check=True)
print("\n=== Running full terraform destroy ===")
rc = _run(f"terraform destroy -auto-approve -var-file={shlex.quote(str(tfvars))}", cwd=stack_dir, check=False)

if rc == 0:
    print("\n=== Teardown complete: All Terraform-managed resources destroyed ===")
else:
    print(f"\n=== Teardown finished with errors (exit_code={rc}) ===")
    print("Check the terraform output above; if an LB/VCN dependency is lingering,")
    print("wait a couple of minutes for the LB to fully detach and re-run this cell once.")
